In [516]:
import pandas as pd

In [539]:
data = pd.read_csv("data/Smart Grid - Sheet1 (4).csv")

data

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,I_level,S_level,Business impact,Data Lifetime,Contains PII?,Exposure,Device,Firmware upgradable,Crypto modifiable,Vendor Support of PQC
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,H,M,High,5–10y,No,Internet / external B2B (High),"Market systems, EMS/market servers",Y,Y,N/Unknown
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,H,M,High,~5–10y,No,Utility WAN / extranet (Medium),EMS / DMS application servers,Y,Y,N/Unknown
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,H,L,High,10y+,No / Possible,Utility WAN (Medium),MDMS servers,Y,Y,N/Unknown
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,H,H,High,"Telemetry days, logs 5+ years",No,Utility WAN (Medium),"SCADA master, automation headend servers",Y,Y,N/Unknown
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,H,H,High,~5–10y,No,Utility WAN (Medium),DMS & automation headend servers,Y,Y,N/Unknown
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,H,L–M,High,7–10y,Yes,Utility WAN / data centre network (Medium),"AMI headend, MDMS servers",Y,Y,N/Unknown
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,H,H,High,"Years (event logs, switching records)",No,Long-distance WAN / wireless (High),Automation frontend (RTU/IED gateway),Maybe,Maybe,N
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,H,H,High,Logs 5+ years,No,Substation LAN (Low–Medium),Substation controller / bay controllers (Resou...,Maybe,Maybe,N
8,C9,Primary Substation Node ↔ Local I/O,Primary Substation – Process,L1–L0,"Modbus RS485, IEC 61850 Ethernet, analog loops",None,No,NaN,No,None (physical isolation only),...,H,H,High,"Real-time, logs <1y",No,Local wired network (Low),"IEDs, sensors, actuators (Resource Constrained)",Limited/Maybe,No,N
9,C10,Secondary Substation Node / Concentrator ↔ Sma...,Secondary Substation (med/low volt.) – Custome...,L2–L1,"G3-PLC, PRIME, CX1, LonTalk over PLC; DLMS/COS...",DLMS/COSEM security (AES),Yes,PSK / shared symmetric keys (per meter or per ...,Yes (DLMS MAC),EndToEnd (between meter and headend/concentrator),...,M–H,L–M,High,7–10y,Yes,Field PLC over public LV grid (High),"Smart meters (Resource Constrained), concentra...",Often Y,Limited,N


In [540]:
data["Sec_Protocol"][0]

'TLS 1.2/1.3 (sometimes plus VPN/IPsec)'

In [541]:
if data["Sec_Protocol"][0] == "TLS 1.2/1.3 (sometimes plus VPN/IPsec)":
    print(2)

2


In [542]:
import re

protocols = ["TLS","DLTS","IPsec","MACsec","SSH","DLMS","COSEM","OPC UA", "DNP3","BACnet/SC", "HTTPS"]

pattern = r"\b(?:%s)\b" % "|".join(map(re.escape, protocols))

mask = data["Sec_Protocol"].astype(str).str.contains(pattern, case=False, na=False)


In [543]:
mask_1 = data["Sec_Placement"].astype(str).str.contains(r"\bWAN\b", case=False, na=False)

mask_1

0     False
1     False
2     False
3     False
4     False
5     False
6      True
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
Name: Sec_Placement, dtype: bool

In [544]:
protocols_placement = ["EndToEnd","GatewayOnly"]

protocols_pattern = r"\b(?:%s)\b" % "|".join(map(re.escape, protocols))

mask_7 = data["Sec_Placement"].astype(str).str.contains(r"\bGatewayOnly\b", case = False, na =  False)

mask_7



0     False
1     False
2     False
3      True
4      True
5     False
6      True
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
Name: Sec_Placement, dtype: bool

In [545]:
mask_2 = data["Data Lifetime"].astype(str).str.contains(
    r"(?<!\w)10y(?:\+)?(?!\w)",
    case=False,
    na=False
)
mask_2

0      True
1      True
2      True
3     False
4      True
5      True
6     False
7     False
8     False
9      True
10    False
11    False
12    False
13    False
14    False
15    False
Name: Data Lifetime, dtype: bool

In [546]:
mask_3 = data["Device"].astype(str).str.contains(r"\bResource Constrained\b", case=False, na=False)

mask_3

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7      True
8      True
9      True
10    False
11    False
12    False
13    False
14    False
15     True
Name: Device, dtype: bool

In [547]:
mask_4 = data["Exposure"].astype(str).str.contains(r"\blow\b", case = False , na = False)

mask_4

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7      True
8      True
9     False
10    False
11    False
12    False
13     True
14    False
15     True
Name: Exposure, dtype: bool

### MAIN Algorithm coding

In [557]:
endpoit_cap = " "
deplpoyable_CAT = " "

CAT = []

for i in range(len(mask)):

    if mask[i] == True:

        default_final_vote = default_vote(i)
        impact_final_vote =  impact_vote(i)
        lifetime_final_vote = lifetime_vote(i)
        exposure_final_vote = exposure_vote(i)

        required_cat = max(int(default_vote(i)[4:]), int(impact_vote(i)[4:]), int(lifetime_vote(i)[4:]),int(exposure_vote(i)[4:]))

        # print(required_cat)

        # deploy(required_cat,i,endpoit_cap)

        # print(deploy)

        if mask_3[i] == True:
            endpoit_cap = "CAT-1"
            # print("ok")
            if required_cat > int(endpoit_cap[4:]):
                # print("ook okk")
                if mask_7[i] == True:
                   deplpoyable_CAT = int(required_cat)
                   
                else:
        #         # print("Try to to use gateway only placement")
        #         # print("optionally use longer-lived sessions/less frequent handshakes")
                    deplpoyable_CAT = int(endpoit_cap[4:])
                    
        else:
            deplpoyable_CAT= required_cat

        print(f"This is deployable",deplpoyable_CAT)
        CAT.append(deplpoyable_CAT)
    

        
    else:
        print("can you upgrade the protocol to a secure veriosn? 1. Yes , 2. No")
        # choiche = input("please write yes or not" )
        if mask_1[i] == True or data["Purdue Layer"][i] == "L2–L2" :

            # print("ok")
            
            default_final_vote = default_vote(i)
            impact_final_vote =  impact_vote(i)
            lifetime_final_vote = lifetime_vote(i)
            exposure_final_vote = exposure_vote(i)

            required_cat = max(int(default_vote(i)[4:]), int(impact_vote(i)[4:]), int(lifetime_vote(i)[4:]),int(exposure_vote(i)[4:]))

            # print(required_cat)
            
            # deploy(required_cat,i,endpoit_cap)

            # print(deploy)

            if mask_3[i] == True:
                endpoit_cap = "CAT-1"
                # print("ok")
                if required_cat > int(endpoit_cap[4:]):
                    # print("ook okk")
                    if mask_7[i] == True:
                        deplpoyable_CAT = int(required_cat)
                        
                    else:
            # #         #print("Try to to use gateway only placement")
            # #         # print("optionally use longer-lived sessions/less frequent handshakes")
                        deplpoyable_CAT = int(endpoit_cap[4:])
                        
            else:
                deplpoyable_CAT= required_cat

        
            print(f"This is deployable",deplpoyable_CAT)
            CAT.append(deplpoyable_CAT)
                
            

            

        else:
            default_1 ="PQC unprotected"
            print(default_1)
            CAT.append(default_1)

print(CAT)
print(len(CAT))


# data["Deployed CAT"]


This is deployable 5
This is deployable 5
This is deployable 5
This is deployable 3
This is deployable 5
This is deployable 5
This is deployable 3
can you upgrade the protocol to a secure veriosn? 1. Yes , 2. No
This is deployable 1
can you upgrade the protocol to a secure veriosn? 1. Yes , 2. No
PQC unprotected
This is deployable 1
This is deployable 3
can you upgrade the protocol to a secure veriosn? 1. Yes , 2. No
PQC unprotected
This is deployable 3
This is deployable 3
This is deployable 3
can you upgrade the protocol to a secure veriosn? 1. Yes , 2. No
PQC unprotected
[5, 5, 5, 3, 5, 5, 3, 1, 'PQC unprotected', 1, 3, 'PQC unprotected', 3, 3, 3, 'PQC unprotected']
16


In [558]:
df = data 

df.head()


,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,I_level,S_level,Business impact,Data Lifetime,Contains PII?,Exposure,Device,Firmware upgradable,Crypto modifiable,Vendor Support of PQC
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,H,M,High,5–10y,No,Internet / external B2B (High),"Market systems, EMS/market servers",Y,Y,N/Unknown
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,H,M,High,~5–10y,No,Utility WAN / extranet (Medium),EMS / DMS application servers,Y,Y,N/Unknown
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,H,L,High,10y+,No / Possible,Utility WAN (Medium),MDMS servers,Y,Y,N/Unknown
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,H,H,High,"Telemetry days, logs 5+ years",No,Utility WAN (Medium),"SCADA master, automation headend servers",Y,Y,N/Unknown
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,H,H,High,~5–10y,No,Utility WAN (Medium),DMS & automation headend servers,Y,Y,N/Unknown


In [559]:
df["Possible Deployed CAT"] = CAT

df["Possible Deployed CAT"] = pd.Series(CAT, index= df.index)

df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,S_level,Business impact,Data Lifetime,Contains PII?,Exposure,Device,Firmware upgradable,Crypto modifiable,Vendor Support of PQC,Possible Deployed CAT
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,M,High,5–10y,No,Internet / external B2B (High),"Market systems, EMS/market servers",Y,Y,N/Unknown,5
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,M,High,~5–10y,No,Utility WAN / extranet (Medium),EMS / DMS application servers,Y,Y,N/Unknown,5
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,L,High,10y+,No / Possible,Utility WAN (Medium),MDMS servers,Y,Y,N/Unknown,5
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,H,High,"Telemetry days, logs 5+ years",No,Utility WAN (Medium),"SCADA master, automation headend servers",Y,Y,N/Unknown,3
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,H,High,~5–10y,No,Utility WAN (Medium),DMS & automation headend servers,Y,Y,N/Unknown,5
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,L–M,High,7–10y,Yes,Utility WAN / data centre network (Medium),"AMI headend, MDMS servers",Y,Y,N/Unknown,5
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,H,High,"Years (event logs, switching records)",No,Long-distance WAN / wireless (High),Automation frontend (RTU/IED gateway),Maybe,Maybe,N,3
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,H,High,Logs 5+ years,No,Substation LAN (Low–Medium),Substation controller / bay controllers (Resou...,Maybe,Maybe,N,1
8,C9,Primary Substation Node ↔ Local I/O,Primary Substation – Process,L1–L0,"Modbus RS485, IEC 61850 Ethernet, analog loops",None,No,NaN,No,None (physical isolation only),...,H,High,"Real-time, logs <1y",No,Local wired network (Low),"IEDs, sensors, actuators (Resource Constrained)",Limited/Maybe,No,N,PQC unprotected
9,C10,Secondary Substation Node / Concentrator ↔ Sma...,Secondary Substation (med/low volt.) – Custome...,L2–L1,"G3-PLC, PRIME, CX1, LonTalk over PLC; DLMS/COS...",DLMS/COSEM security (AES),Yes,PSK / shared symmetric keys (per meter or per ...,Yes (DLMS MAC),EndToEnd (between meter and headend/concentrator),...,L–M,High,7–10y,Yes,Field PLC over public LV grid (High),"Smart meters (Resource Constrained), concentra...",Often Y,Limited,N,1


In [548]:
def default_vote(j):

    # if mask_2[j] == True:
    #     default = "CAT-5"
        #print("ok")
    if mask_3[j] == True and mask_4[j] == True:
        default = "CAT-1"
        #print(default)
    else:
        default = "CAT-3"

    return default

In [549]:
#classify use case type 
#Experiment

for i in range(len(mask)):
        #print("ok")
    if mask_3[i] == True and mask_4[i] == True:
        default = "CAT-1"
        #print(default)
    else:
        default = " CAT-3" 
        #print(default)
    
    print(default)

 CAT-3
 CAT-3
 CAT-3
 CAT-3
 CAT-3
 CAT-3
 CAT-3
CAT-1
CAT-1
 CAT-3
 CAT-3
 CAT-3
 CAT-3
 CAT-3
 CAT-3
CAT-1


In [550]:
#impact vote 

def impact_vote(k):
    for i in range(len(mask)):
        if data["C_level"][k]  == "L" and data["A_level"][k] == "L" and data["I_level"][k] == "L" and data["Business impact"][k] == "L":
            impact_vote = "CAT-1"
        else:
            impact_vote = "CAT-3"
    
    return impact_vote

In [551]:
#exposure vote 

def exposure_vote(l):

    protocols = ["Low"]

    pattern_1 = r"\b(?:%s)\b" % "|".join(map(re.escape, protocols))

    mask_5 = data["Exposure"].astype(str).str.contains(pattern_1, case = False , na =  False )
    
    if mask_5[l] == True:
        exposure = "CAT-1"
    else:
        exposure = "CAT-3"

    return exposure


In [552]:
#lifetimevote 

def lifetime_vote(m):

    if mask_2[m] == True: 
        lifetime_vote = "CAT-5"
    else:
        lifetime_vote = "CAT-3"
        
    return lifetime_vote
        
    

In [553]:
### end to end to gateway is available  already


# required_cat = max(int(impact_vote[4:],int(default[4:],int(exposure_vote[4:],int(lifetime_vote[4:])))))

# required_cat






In [554]:
#check deployablitiy
endpoit_cap = "CAT-1"

def deploy(required,o,endpoit_cap):
    if mask_3[o] == True:
        endpoit_cap = "CAT-1"
        if required > int(endpoit_cap[4:]):
            if mask_7[o] == True:
                deplpoyable_CAT = int(endpoit_cap[4:])
            else:
                print("Try to to use gateway only placement")
                print("optionally use longer-lived sessions/less frequent handshakes")
    else:
        deplpoyable_CAT= endpoit_cap
    
    return deplpoyable_CAT
        
            
            


In [555]:
if mask.all():
    print(2)

In [556]:
print(data.loc[mask, "Sec_Protocol"])


0                TLS 1.2/1.3 (sometimes plus VPN/IPsec)
1     TLS 1.2/1.3 (for WebService), TLS for SMTP + o...
2                           TLS 1.2/1.3 (often via VPN)
3     IPsec ESP (site-to-site), sometimes None in le...
4                              IPsec ESP (site-to-site)
5                                           TLS 1.2/1.3
6        IPsec ESP (typical), sometimes None / MAC-only
9                             DLMS/COSEM security (AES)
10                              TLS 1.2 (or vendor TLS)
12                            TLS 1.2 (sometimes + VPN)
13    TLS 1.2 (IEC 15118 sessions), None for pure 61...
14                                          TLS 1.2/1.3
Name: Sec_Protocol, dtype: str


### Profile Recommmender 


In [560]:
#Initial Logic 

df["Possible Deployed CAT"]


for i in range(len(mask)):
    if df["Possible Deployed CAT"][i] == "PQC unprotected":
        print("ok")
    else:
        if df["Has_Enc"][i] != "NO":


            print(df["Has_Enc"][i])
             

Yes
Yes
Yes
Yes (where IPsec is used), No (legacy)
Yes
Yes
Yes (when IPsec enabled)
Partial (Yes where MACsec/VPN used, No otherwise)
ok
Yes
Yes
ok
Yes
Partial
Yes
ok


In [561]:
xls = pd.ExcelFile('data/oqs_speed_parsed_nist_sizes.xlsx')


df_cpu_meta = pd.read_excel(xls,"SIG_meta")
df_cpu_benchmarking = pd.read_excel(xls,"SIG_wide")
df_cpu_benchmarking_kem = pd.read_excel(xls,"KEM_wide")

In [562]:
xls_mcu = pd.ExcelFile("data/pqm4_size_nist_sizes.xlsx")

df_mcu_meta = pd.read_excel(xls_mcu,"README1")
df_mcu_benchmarking = pd.read_excel(xls_mcu,"SIG_table")
df_mcu_benchmarking_kem = pd.read_excel(xls_mcu,"KEM_table")

df_mcu_benchmarking_kem

,Algorithm,NIST_Category,PK_bytes,SK_bytes,CT_bytes,Executions,Implementation,.text [bytes],.data [bytes],.bss [bytes],...,Implementation.2,Key Generation [cycles] (mean),Key Generation [cycles] (min),Key Generation [cycles] (max),Encapsulation [cycles] (mean),Encapsulation [cycles] (min),Encapsulation [cycles] (max),Decapsulation [cycles] (mean),Decapsulation [cycles] (min),Decapsulation [cycles] (max)
0,bikel1,1,1541,5223,1573,NaN,m4f,181088,24,49,...,m4f,27382389,27382366,27382406,3363252,3363238,3363291,56744036,56744013,56744062
1,bikel1,1,1541,5223,1573,NaN,opt,34451,24,1,...,opt,76272765,76272741,76272809,5295900,5295877,5295931,138514606,138514541,138514711
2,bikel3,3,3083,10105,3115,NaN,m4f,198034,24,49,...,m4f,66285221,66285184,66285235,8441498,8441478,8441521,150282952,150282849,150282974
3,bikel3,3,3083,10105,3115,NaN,opt,43091,24,1,...,opt,248083316,248083286,248083345,16405238,16405236,16405241,423262047,423261952,423262086
4,hqc-128,1,2249,2305,4433,NaN,clean,18628,0,0,...,clean,52705201,52705180,52705224,105650897,105650877,105650927,159569179,159569176,159569183
5,hqc-192,3,4522,4586,8978,NaN,clean,21104,0,0,...,clean,161458617,161458590,161458638,323146261,323146250,323146292,486156251,486156214,486156266
6,hqc-256,5,7245,7317,14421,NaN,clean,26260,0,0,...,clean,295934078,295934057,295934104,591853870,591853850,591853898,891163005,891162988,891163038
7,ml-kem-1024,5,1568,3168,1568,NaN,clean,6160,0,0,...,clean,1536343,1535750,1536698,1708071,1707476,1708427,2020327,2019721,2020672
8,ml-kem-1024,5,1568,3168,1568,NaN,m4fspeed,16916,0,0,...,m4fspeed,1018976,1014877,1026934,1031565,1027454,1039544,1094008,1089897,1101987
9,ml-kem-1024,5,1568,3168,1568,NaN,m4fstack,14016,0,0,...,m4fstack,1020202,1017478,1029553,1037953,1035260,1047298,1100982,1098251,1110327


In [563]:
df_cpu_meta

,Key,Value
0,Target platform,arm64-Darwin-23.0.0
1,Compiler,clang (15.0.0 (clang-1500.3.9.4))
2,Compile options,"[-march=armv8-a+crypto;-Wa,--noexecstack;-O3;-..."
3,OQS version,"0.14.1-dev (major: 0, minor: 14, patch: 1, pre..."
4,Git commit,6e6ffa5082f3191fdf4163cbcd75de623a097f65
5,OpenSSL enabled,Yes (OpenSSL 3.6.0 1 Oct 2025)
6,AES,OpenSSL
7,SHA-2,OpenSSL
8,SHA-3,C
9,OQS build flags,OQS_DIST_BUILD OQS_LIBJADE_BUILD OQS_OPT_TARGE...


In [564]:
df_cpu_benchmarking

,Algorithm,NIST_Category,PK_bytes,SK_bytes,SIG_bytes,keypair__Time_us_mean,sign__Time_us_mean,verify__Time_us_mean,keypair__Time_us_stdev,sign__Time_us_stdev,verify__Time_us_stdev,keypair__HighPrec_ns_mean,sign__HighPrec_ns_mean,verify__HighPrec_ns_mean,keypair__HighPrec_ns_stdev,sign__HighPrec_ns_stdev,verify__HighPrec_ns_stdev,keypair__Iterations,sign__Iterations,verify__Iterations
0,Dilithium2,2.0,1312,2528,2420,21.133,64.181,20.246,29.641,40.509,1.279,21101,64150,20216,29640,40508,1280,141960,46745,148179
1,Dilithium3,3.0,1952,4000,3293,46.921,100.135,31.969,2.423,76.689,9.225,46889,100102,31939,2425,76691,9225,63938,29961,93841
2,Dilithium5,5.0,2592,4864,4595,53.106,119.250,50.323,2.517,61.300,1.361,53076,119213,50292,2518,61301,1357,56491,25158,59615
3,Falcon-1024,5.0,1793,2305,1462,16699.806,308.601,48.131,3973.441,5.258,3.340,16699772,308565,48101,3973438,5259,3340,180,9722,62330
4,Falcon-512,1.0,897,1281,752,5191.730,156.820,24.786,1163.886,5.776,112.612,5191692,156787,24757,1163889,5777,112612,578,19131,121036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,cross-rsdpg-192-fast,3.0,83,48,26772,10.534,790.406,474.641,2.145,6.919,280.571,10504,790361,474605,2142,6915,280561,284787,3796,6321
220,cross-rsdpg-192-small,3.0,83,48,20452,10.532,2037.699,1077.481,2.312,24.596,9.815,10501,2037646,1077437,2312,24605,9819,284854,1473,2785
221,cross-rsdpg-256-balanced,5.0,106,64,40100,17.257,1892.461,999.861,2.282,11.224,6.527,17227,1892404,999816,2282,11229,6527,173844,1586,3001
222,cross-rsdpg-256-fast,5.0,106,64,48102,17.269,1477.215,871.743,2.309,47.237,10.625,17239,1477154,871698,2308,47242,10624,173724,2031,3442


In [565]:
bucket_std = ["dilithium", "falcon", "sphincs"]  # use sphincs (without +) to match all variants

mask_std = df_cpu_benchmarking["Algorithm"].astype(str).str.lower().str.contains(
    "|".join(bucket_std), na=False
)
mask_std


0       True
1       True
2       True
3       True
4       True
       ...  
219    False
220    False
221    False
222    False
223    False
Name: Algorithm, Length: 224, dtype: bool

In [566]:
df_cpu_benchmarking_kem

,Algorithm,NIST_Category,PK_bytes,SK_bytes,CT_bytes,decaps__Time_us_mean,encaps__Time_us_mean,keygen__Time_us_mean,decaps__Time_us_stdev,encaps__Time_us_stdev,keygen__Time_us_stdev,decaps__HighPrec_ns_mean,encaps__HighPrec_ns_mean,keygen__HighPrec_ns_mean,decaps__HighPrec_ns_stdev,encaps__HighPrec_ns_stdev,keygen__HighPrec_ns_stdev,decaps__Iterations,encaps__Iterations,keygen__Iterations
0,BIKE-L1,1.0,1541,5223,1573,5160.555,371.287,6785.621,96.660,67.074,1273.502,5160527,371260,6785580,96662,67076,1273504,582,8080,443
1,BIKE-L3,3.0,3083,10105,3115,16316.946,1123.900,20761.193,1250.906,661.329,416.882,16316897,1123850,20761143,1250903,661329,416875,184,2670,145
2,BIKE-L5,5.0,5122,16494,5154,39993.868,2781.734,67176.889,403.536,105.908,44072.671,39993829,2781692,67176823,403560,105899,44072676,76,1079,45
3,Classic-McEliece-348864,NaN,261120,6492,96,19597.240,20.310,61499.163,412.438,3.405,38017.277,19597207,20281,61499141,412457,3406,38017297,154,147709,49
4,Classic-McEliece-348864f,NaN,261120,6492,96,20271.405,20.550,33609.133,7358.979,3.508,578.080,20271388,20520,33609088,7358981,3508,578058,148,145987,90
5,Classic-McEliece-460896,NaN,524160,13608,156,42217.139,40.745,209264.533,11703.982,19.848,130443.085,42217095,40714,209264401,11703877,19846,130443178,72,73629,15
6,Classic-McEliece-460896f,NaN,524160,13608,156,40510.813,38.415,99466.484,93.778,13.517,568.475,40510788,38384,99466479,93789,13518,568514,75,78095,31
7,Classic-McEliece-6688128,NaN,1044992,13932,208,78033.641,84.176,389389.875,634.978,26.821,227787.859,78033592,84144,389389888,635006,26822,227787842,39,35640,8
8,Classic-McEliece-6688128f,NaN,1044992,13932,208,78985.000,97.126,174617.889,1412.820,178.424,604.007,78984994,97094,174617771,1412827,178420,604097,38,30889,18
9,Classic-McEliece-6960119,NaN,1047319,13948,194,76684.250,726.852,366167.333,753.504,374.246,204083.105,76684218,726819,366167239,753502,374249,204083203,40,4128,9


In [567]:
for i in range(len(mask_std)):
    if mask_std[i] == True:
        print(df_cpu_benchmarking["Algorithm"][i])

Dilithium2
Dilithium3
Dilithium5
Falcon-1024
Falcon-512
Falcon-padded-1024
Falcon-padded-512
SPHINCS+-SHA2-128f-simple
SPHINCS+-SHA2-128s-simple
SPHINCS+-SHA2-192f-simple
SPHINCS+-SHA2-192s-simple
SPHINCS+-SHA2-256f-simple
SPHINCS+-SHA2-256s-simple
SPHINCS+-SHAKE-128f-simple
SPHINCS+-SHAKE-128s-simple
SPHINCS+-SHAKE-192f-simple
SPHINCS+-SHAKE-192s-simple
SPHINCS+-SHAKE-256f-simple
SPHINCS+-SHAKE-256s-simple


In [568]:
# protocols = ["TLS","DLTS","IPsec","MACsec","SSH","DLMS","COSEM","OPC UA", "DNP3","BACnet/SC", "HTTPS"]

# pattern = r"\b(?:%s)\b" % "|".join(map(re.escape, protocols))

# mask = data["Sec_Protocol"].astype(str).str.contains(pattern, case=False, na=False)



bucket_std_low_layer = ["L0", "L1", "L2"] 

pattern_low = r"\b(?:%s)\b" % "|".join(map(re.escape, bucket_std_low_layer)) 

mask_layer_low= data["Purdue Layer"].astype(str).str.contains(pattern_low, case=False, na=False)


# mask_layer_low = df["Purdue Layer"].astype(str).str.lower().str.contains(
#     "|".join(bucket_std_low_layer), na=False
# )

bucket_layer_high = ["L4", "L5"]

pattern_high = r"\b(?:%s)\b" % "|".join(map(re.escape, bucket_layer_high)) 

mask_layer_high= data["Purdue Layer"].astype(str).str.contains(pattern_high, case=False, na=False)


# mask_layer_high = df["Purdue Layer"].astype(str).str.lower().str.contains(
#     "|".join(bucket_layer_high), na=False
# )

bucket_layer_medium = ["L3"]

pattern_medium = r"\b(?:%s)\b" % "|".join(map(re.escape, bucket_layer_medium)) 

mask_layer_medium= data["Purdue Layer"].astype(str).str.contains(pattern_medium, case=False, na=False)

# mask_layer_medium = df["Purdue Layer"].astype(str).str.lower().str.contains(
#     "|".join(bucket_layer_medium), na=False
# )


# mask_layer_high
# mask_layer_medium
# # mask_layer_low

In [569]:
bucket_protocol_layer = ["Ethernet","Wireless", "VPN", "TLS" ,"WebService" , "Internet"] 

pattern_protocol = r"\b(?:%s)\b" % "|".join(map(re.escape, bucket_protocol_layer)) 

mask_protocol_high = df["Protocols"].astype(str).str.contains(pattern_protocol, case=False, na=False)

mask_protocol_high

0      True
1      True
2      True
3      True
4      True
5      True
6      True
7      True
8      True
9     False
10     True
11     True
12     True
13    False
14     True
15     True
Name: Protocols, dtype: bool

In [570]:
bucket_device = ["Servers", "gateway"] 

pattern_device = r"\b(?:%s)\b" % "|".join(map(re.escape, bucket_device )) 

mask_device= df["Device"].astype(str).str.contains(pattern_device, case=False, na=False)

mask_device

0      True
1      True
2      True
3      True
4      True
5      True
6      True
7     False
8     False
9     False
10     True
11    False
12    False
13    False
14     True
15    False
Name: Device, dtype: bool

In [571]:
mask_3

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7      True
8      True
9      True
10    False
11    False
12    False
13    False
14    False
15     True
Name: Device, dtype: bool

In [572]:
df_mcu_benchmarking_kem["Decapsulation [cycles] (mean)"]

0      56744036
1     138514606
2     150282952
3     423262047
4     159569179
5     486156251
6     891163005
7       2020327
8       1094008
9       1100982
10       888653
11       428167
12       430202
13      1387984
14       707827
15       714194
Name: Decapsulation [cycles] (mean), dtype: int64

In [573]:
#utility score 
#Encryption utility score experiemental


#latency Buckets 

cpu_Total = []
bytes_total = []

cpu_Total_1 = []
bytes_total_1 =[]

cpu_Total_5 = []
bytes_total_5 =[]

for j in range(len(df_cpu_benchmarking_kem["NIST_Category"])):
     if df_cpu_benchmarking_kem["NIST_Category"][j] == 3:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking_kem["decaps__Time_us_mean"][j] + df_cpu_benchmarking_kem["encaps__Time_us_mean"][j] 
         cpu_Total.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking_kem["PK_bytes"][j] + df_cpu_benchmarking_kem["CT_bytes"][j]

         bytes_total.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

     if df_cpu_benchmarking_kem["NIST_Category"][j] == 1:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking_kem["decaps__Time_us_mean"][j] + df_cpu_benchmarking_kem["encaps__Time_us_mean"][j] 
         cpu_Total_1.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking_kem["PK_bytes"][j] + df_cpu_benchmarking_kem["CT_bytes"][j]

         bytes_total_1.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })
    
     if df_cpu_benchmarking_kem["NIST_Category"][j] == 5:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking_kem["decaps__Time_us_mean"][j] + df_cpu_benchmarking_kem["encaps__Time_us_mean"][j] 
         cpu_Total_5.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking_kem["PK_bytes"][j] + df_cpu_benchmarking_kem["CT_bytes"][j]

         bytes_total_5.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })



mcu_total = []
bytes_total_mcu = []
ram_peak = []

mcu_total_1 = []
bytes_total_mcu_1 = []
ram_peak_1 = []

mcu_total_5 = []
bytes_total_mcu_5 = []
ram_peak_5 = []

for k in range(len(df_mcu_benchmarking_kem["NIST_Category"])):
     if df_cpu_benchmarking_kem["NIST_Category"][k] == 3:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking_kem["Encapsulation [cycles] (mean)"][k] + df_mcu_benchmarking_kem["Decapsulation [cycles] (mean)"][k] 
         mcu_total.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking_kem["PK_bytes"][k] + df_mcu_benchmarking_kem["CT_bytes"][k]

         bytes_total_mcu.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking_kem["Encapsulation [bytes]"][k], df_mcu_benchmarking_kem["Decapsulation [bytes]"][k])

         ram_peak.append({
             "Name":df_mcu_benchmarking_kem.loc[k, "Algorithm"],
             "value": ram
         })
    
     if df_cpu_benchmarking_kem["NIST_Category"][k] == 1:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking_kem["Encapsulation [cycles] (mean)"][k] + df_mcu_benchmarking_kem["Decapsulation [cycles] (mean)"][k] 
         mcu_total_1.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking_kem["PK_bytes"][k] + df_mcu_benchmarking_kem["CT_bytes"][k]

         bytes_total_mcu_1.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking_kem["Encapsulation [bytes]"][k], df_mcu_benchmarking_kem["Decapsulation [bytes]"][k])

         ram_peak_1.append({
             "Name":df_mcu_benchmarking_kem.loc[k, "Algorithm"],
             "value": ram
         })
        
     if df_cpu_benchmarking_kem["NIST_Category"][k] == 5:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking_kem["Encapsulation [cycles] (mean)"][k] + df_mcu_benchmarking_kem["Decapsulation [cycles] (mean)"][k] 
         mcu_total_5.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking_kem["PK_bytes"][k] + df_mcu_benchmarking_kem["CT_bytes"][k]

         bytes_total_mcu_5.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking_kem["Encapsulation [bytes]"][k], df_mcu_benchmarking_kem["Decapsulation [bytes]"][k])

         ram_peak_5.append({
             "Name":df_mcu_benchmarking_kem.loc[k, "Algorithm"],
             "value": ram
         })


#us
Strict = 10000 
Medium = 50000 
Loose = 200000 


#allowed packets 

N_pkts_server = 10  #server/gateway 
N_pkts_emb = 6  #Embedded 
N_pkts_cons = 3 #constrained 


byte_ref_etherenet_server = 1500 * N_pkts_server
byte_ref_mtu_server = 512 * N_pkts_server 

byte_ref_etherenet_cons = 1500 * N_pkts_cons
byte_ref_mtu_cons = 512 * N_pkts_cons

byte_ref_etherenet_emb = 1500 * N_pkts_emb
byte_ref_mtu_emb = 512 * N_pkts_emb

f = 120 #MHZ
Strict_mcu = 1200000
Medium_mcu = 6000000
Loose_MCU = 24000000


for i in range(len(mask)):
    if df["Possible Deployed CAT"][i] == "PQC unprotected":
        print("ok")
    else:
        if mask_3[i] == True: #checking resource constrained
            if df["Has_Enc"][i] != "NO":
                if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #nist catagory - 1
                if df["Possible Deployed CAT"][i] == 1:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #NIST catagory  -5         

                if df["Possible Deployed CAT"][i] == 5:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

        elif mask_device[i] == True:
             if df["Has_Enc"][i] != "NO":
                if df["Possible Deployed CAT"][i] == 3:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #nist catagory - 1 

                if df["Possible Deployed CAT"][i] == 1:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #NIST catagory - 5         
                if df["Possible Deployed CAT"][i] == 5:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

        else: #checking embeded
            if df["Has_Enc"][i] != "NO":
                if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 1:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 5:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    

                

L5–L4
C1
[{'Name': 'BIKE-L5', 'utility': 0.355234607}, {'Name': 'Kyber1024', 'utility': 0.0628255635}, {'Name': 'ML-KEM-1024', 'utility': 0.06288374399999999}]
L4–L3
C2
[{'Name': 'BIKE-L5', 'utility': 0.804378428}, {'Name': 'Kyber1024', 'utility': 0.063142254}, {'Name': 'ML-KEM-1024', 'utility': 0.063374976}]
L4–L3
C2
[{'Name': 'BIKE-L5', 'utility': 0.355234607}, {'Name': 'Kyber1024', 'utility': 0.0628255635}, {'Name': 'ML-KEM-1024', 'utility': 0.06288374399999999}]
L4–L3
C3
[{'Name': 'BIKE-L5', 'utility': 0.804378428}, {'Name': 'Kyber1024', 'utility': 0.063142254}, {'Name': 'ML-KEM-1024', 'utility': 0.063374976}]
L4–L3
C3
[{'Name': 'BIKE-L5', 'utility': 0.355234607}, {'Name': 'Kyber1024', 'utility': 0.0628255635}, {'Name': 'ML-KEM-1024', 'utility': 0.06288374399999999}]
L3–L3
C4
[{'Name': 'BIKE-L3', 'utility': 0.368131844}, {'Name': 'Kyber768', 'utility': 0.045752746000000004}, {'Name': 'ML-KEM-768', 'utility': 0.045883394}]
L3–L3
C5
[{'Name': 'BIKE-L5', 'utility': 0.804378428}, {'Nam

In [574]:
#clean version 
#no repeated function xDDDDDDDDD


# ============================================================
# CLEAN version: CAT 1/3/5 + server/constrained/embedded
# prints once per ID + writes results into df columns
# ============================================================

# ---------- 0) helpers ----------
def safe_int_cat(x):
    """Convert Possible Deployed CAT to int if it's 1/3/5; otherwise return None."""
    if isinstance(x, (int, float)) and x in (1, 3, 5):
        return int(x)
    if isinstance(x, str):
        x = x.strip()
        if x.isdigit():
            v = int(x)
            if v in (1, 3, 5):
                return v
    return None

import pandas as pd

def build_server_maps(df_cpu_benchmarking_kem, cats=(1,3,5)):
    maps = {c: {"cpu_us": {}, "bytes": {}} for c in cats}

    for j in range(len(df_cpu_benchmarking_kem)):
        cat_raw = df_cpu_benchmarking_kem.loc[j, "NIST_Category"]

        # skip missing
        if pd.isna(cat_raw):
            continue

        # safe convert
        try:
            cat = int(float(cat_raw))
        except Exception:
            continue

        if cat not in maps:
            continue

        alg = df_cpu_benchmarking_kem.loc[j, "Algorithm"]
        cpu_us = float(df_cpu_benchmarking_kem.loc[j, "decaps__Time_us_mean"]) + float(df_cpu_benchmarking_kem.loc[j, "encaps__Time_us_mean"])
        b = float(df_cpu_benchmarking_kem.loc[j, "PK_bytes"]) + float(df_cpu_benchmarking_kem.loc[j, "CT_bytes"])

        maps[cat]["cpu_us"][alg] = cpu_us
        maps[cat]["bytes"][alg]  = b

    return maps

def build_mcu_maps(df_mcu_benchmarking_kem, cats=(1,3,5)):
    maps = {c: {"cycles": {}, "bytes": {}, "ram": {}} for c in cats}

    for k in range(len(df_mcu_benchmarking_kem)):
        cat_raw = df_mcu_benchmarking_kem.loc[k, "NIST_Category"]

        if pd.isna(cat_raw):
            continue

        try:
            cat = int(float(cat_raw))
        except Exception:
            continue

        if cat not in maps:
            continue

        alg = df_mcu_benchmarking_kem.loc[k, "Algorithm"]

        cyc = float(df_mcu_benchmarking_kem.loc[k, "Encapsulation [cycles] (mean)"]) + float(df_mcu_benchmarking_kem.loc[k, "Decapsulation [cycles] (mean)"])
        b = float(df_mcu_benchmarking_kem.loc[k, "PK_bytes"]) + float(df_mcu_benchmarking_kem.loc[k, "CT_bytes"])
        ram_peak = max(float(df_mcu_benchmarking_kem.loc[k, "Encapsulation [bytes]"]),
                       float(df_mcu_benchmarking_kem.loc[k, "Decapsulation [bytes]"]))

        maps[cat]["cycles"][alg] = cyc
        maps[cat]["bytes"][alg]  = b
        maps[cat]["ram"][alg]    = ram_peak

    return maps

def normalize_weights(*ws):
    s = sum(ws)
    if s == 0:
        return ws
    return tuple(w/s for w in ws)

def rank_server(server_map, time_ref_us, bytes_ref, w_cpu=0.7, w_bytes=0.3):
    cpu_map = server_map["cpu_us"]
    b_map   = server_map["bytes"]
    out = []
    for alg in cpu_map.keys():
        cpu_norm = cpu_map[alg] / time_ref_us
        b_norm   = b_map.get(alg, 0.0) / bytes_ref
        u = w_cpu*cpu_norm + w_bytes*b_norm
        out.append({"Name": alg, "utility": u})
    out.sort(key=lambda x: x["utility"])
    return out

def rank_mcu(mcu_map, cycles_ref, bytes_ref, ram_ref, w_cpu=0.6, w_bytes=0.2, w_ram=0.2):
    w_cpu, w_bytes, w_ram = normalize_weights(w_cpu, w_bytes, w_ram)  # safe if you change weights later
    c_map = mcu_map["cycles"]
    b_map = mcu_map["bytes"]
    r_map = mcu_map["ram"]
    out = []
    for alg in c_map.keys():
        cpu_norm = c_map[alg] / cycles_ref
        b_norm   = b_map.get(alg, 0.0) / bytes_ref
        r_norm   = r_map.get(alg, 0.0) / ram_ref
        u = w_cpu*cpu_norm + w_bytes*b_norm + w_ram*r_norm
        out.append({"Name": alg, "utility": u})
    out.sort(key=lambda x: x["utility"])
    return out


# ---------- 1) Build benchmark maps once ----------
server_maps = build_server_maps(df_cpu_benchmarking_kem, cats=(1,3,5))
mcu_maps    = build_mcu_maps(df_mcu_benchmarking_kem, cats=(1,3,5))


# ---------- 2) Refs / constants ----------
STRICT_US, MEDIUM_US, LOOSE_US = 10_000, 50_000, 200_000
STRICT_CYC, MEDIUM_CYC, LOOSE_CYC = 1_200_000, 6_000_000, 24_000_000  # 120MHz * {10,50,200} ms

N_pkts_server, N_pkts_emb, N_pkts_cons = 10, 6, 3

byte_ref_eth_server = 1500 * N_pkts_server
byte_ref_mtu_server = 512  * N_pkts_server

byte_ref_eth_emb = 1500 * N_pkts_emb
byte_ref_mtu_emb = 512  * N_pkts_emb

byte_ref_eth_cons = 1500 * N_pkts_cons
byte_ref_mtu_cons = 512  * N_pkts_cons

DEFAULT_RAM_REF = 131072  # 128KB; replace with df value per-row if you have it


# ---------- 3) Output columns ----------
df["Enc_Ranking_By_Utility"] = ""
df["Enc_Best_Alg"] = ""
df["Enc_Best_Utility"] = 0.0

# print + compute only once per ID
printed_ids = set()

# ---------- 4) Main loop ----------
for i in range(len(df)):
    # skip unprotected
    if df.loc[i, "Possible Deployed CAT"] == "PQC unprotected":
        continue
    if df.loc[i, "Has_Enc"] == "NO":
        continue

    cat = safe_int_cat(df.loc[i, "Possible Deployed CAT"])
    if cat is None:
        continue

    row_id = df.loc[i, "ID"]
    if row_id in printed_ids:
        continue
    printed_ids.add(row_id)

    purdue = df.loc[i, "Purdue Layer"]

    # ---- pick one bucket ----
    if mask_layer_low[i]:
        bucket = "Strict"
        time_ref_us = STRICT_US
        cycles_ref  = STRICT_CYC
    elif mask_layer_medium[i]:
        bucket = "Medium"
        time_ref_us = MEDIUM_US
        cycles_ref  = MEDIUM_CYC
    elif mask_layer_high[i]:
        bucket = "Loose"
        time_ref_us = LOOSE_US
        cycles_ref  = LOOSE_CYC
    else:
        bucket = "Medium"
        time_ref_us = MEDIUM_US
        cycles_ref  = MEDIUM_CYC

    # ---- link type ----
    if mask_protocol_high[i]:
        link = "Ethernet(1500)"
        bytes_ref_server = byte_ref_eth_server
        bytes_ref_emb    = byte_ref_eth_emb
        bytes_ref_cons   = byte_ref_eth_cons
    else:
        link = "lowMTU(512)"
        bytes_ref_server = byte_ref_mtu_server
        bytes_ref_emb    = byte_ref_mtu_emb
        bytes_ref_cons   = byte_ref_mtu_cons

    # ---- device class (your masks) ----
    if mask_3[i] == True:
        device_class = "constrained"
        ranked = rank_mcu(mcu_maps[cat], cycles_ref, bytes_ref_cons, ram_ref=DEFAULT_RAM_REF,
                          w_cpu=0.6, w_bytes=0.2, w_ram=0.2)
    elif mask_device[i] == True:
        device_class = "server"
        ranked = rank_server(server_maps[cat], time_ref_us, bytes_ref_server, w_cpu=0.7, w_bytes=0.3)
    else:
        device_class = "embedded"
        ranked = rank_mcu(mcu_maps[cat], cycles_ref, bytes_ref_emb, ram_ref=DEFAULT_RAM_REF,
                          w_cpu=0.6, w_bytes=0.2, w_ram=0.2)

    # ---- print (same style as before) ----
    print("\n====================================================")
    print(f"ID: {row_id} | CAT:{cat} | Purdue: {purdue} | class: {device_class} | Bucket: {bucket} | Link: {link}")
    print("Ranked utility (lower is better):")
    for r, item in enumerate(ranked, start=1):
        print(f"{r:02d}. {item['Name']}: {item['utility']:.8f}")

    # ---- write back to df for ALL rows with same ID ----
    ranking_str = ", ".join([f'{x["Name"]}:{x["utility"]:.8f}' for x in ranked])
    best = ranked[0]

    df.loc[df["ID"] == row_id, "Enc_Ranking_By_Utility"] = ranking_str
    df.loc[df["ID"] == row_id, "Enc_Best_Alg"] = best["Name"]
    df.loc[df["ID"] == row_id, "Enc_Best_Utility"] = best["utility"]





ID: C1 | CAT:5 | Purdue: L5–L4 | class: server | Bucket: Loose | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber1024: 0.06282556
02. ML-KEM-1024: 0.06288374
03. BIKE-L5: 0.35523461

ID: C2 | CAT:5 | Purdue: L4–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber1024: 0.06314225
02. ML-KEM-1024: 0.06337498
03. BIKE-L5: 0.80437843

ID: C3 | CAT:5 | Purdue: L4–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber1024: 0.06314225
02. ML-KEM-1024: 0.06337498
03. BIKE-L5: 0.80437843

ID: C4 | CAT:3 | Purdue: L3–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04575275
02. ML-KEM-768: 0.04588339
03. BIKE-L3: 0.36813184

ID: C5 | CAT:5 | Purdue: L3–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber1024: 0.06314225
02. ML-KEM-1024: 0.06337498
03. BIKE-L5: 0.80437843

ID

In [575]:
cols_to_drop = ["Enc_Best_Utility", "Best_Alg_CAT3", "Best_Utility_CAT3", "Ranked_Utility_CAT3"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,Data Lifetime,Contains PII?,Exposure,Device,Firmware upgradable,Crypto modifiable,Vendor Support of PQC,Possible Deployed CAT,Enc_Ranking_By_Utility,Enc_Best_Alg
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,5–10y,No,Internet / external B2B (High),"Market systems, EMS/market servers",Y,Y,N/Unknown,5,"Kyber1024:0.06282556, ML-KEM-1024:0.06288374, ...",Kyber1024
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,~5–10y,No,Utility WAN / extranet (Medium),EMS / DMS application servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,10y+,No / Possible,Utility WAN (Medium),MDMS servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,"Telemetry days, logs 5+ years",No,Utility WAN (Medium),"SCADA master, automation headend servers",Y,Y,N/Unknown,3,"Kyber768:0.04575275, ML-KEM-768:0.04588339, BI...",Kyber768
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,~5–10y,No,Utility WAN (Medium),DMS & automation headend servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,7–10y,Yes,Utility WAN / data centre network (Medium),"AMI headend, MDMS servers",Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,"Years (event logs, switching records)",No,Long-distance WAN / wireless (High),Automation frontend (RTU/IED gateway),Maybe,Maybe,N,3,"Kyber768:0.04700373, ML-KEM-768:0.04765697, BI...",Kyber768
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,Logs 5+ years,No,Substation LAN (Low–Medium),Substation controller / bay controllers (Resou...,Maybe,Maybe,N,1,"ml-kem-512:0.48480465, bikel1:72.16349553, hqc...",ml-kem-512
8,C9,Primary Substation Node ↔ Local I/O,Primary Substation – Process,L1–L0,"Modbus RS485, IEC 61850 Ethernet, analog loops",None,No,NaN,No,None (physical isolation only),...,"Real-time, logs <1y",No,Local wired network (Low),"IEDs, sensors, actuators (Resource Constrained)",Limited/Maybe,No,N,PQC unprotected,,
9,C10,Secondary Substation Node / Concentrator ↔ Sma...,Secondary Substation (med/low volt.) – Custome...,L2–L1,"G3-PLC, PRIME, CX1, LonTalk over PLC; DLMS/COS...",DLMS/COSEM security (AES),Yes,PSK / shared symmetric keys (per meter or per ...,Yes (DLMS MAC),EndToEnd (between meter and headend/concentrator),...,7–10y,Yes,Field PLC over public LV grid (High),"Smart meters (Resource Constrained)

In [371]:
# # =========================
# # 0) Build CAT=3 maps ONCE (CPU server in us, MCU in cycles, bytes, RAM peak)
# # =========================

# # ---- SERVER maps (µs + bytes) ----
# cpu_map_server = {}
# bytes_map_server = {}

# for j in range(len(df_cpu_benchmarking_kem)):
#     if df_cpu_benchmarking_kem.loc[j, "NIST_Category"] == 3:
#         name = df_cpu_benchmarking_kem.loc[j, "Algorithm"]
#         cpu_t = float(df_cpu_benchmarking_kem.loc[j, "decaps__Time_us_mean"]) + float(df_cpu_benchmarking_kem.loc[j, "encaps__Time_us_mean"])
#         b_t   = float(df_cpu_benchmarking_kem.loc[j, "PK_bytes"]) + float(df_cpu_benchmarking_kem.loc[j, "CT_bytes"])
#         cpu_map_server[name] = cpu_t
#         bytes_map_server[name] = b_t


# # ---- MCU maps (cycles + bytes + RAM peak bytes) ----
# mcu_cycles_map = {}
# mcu_bytes_map  = {}
# mcu_rampeak_map = {}

# for k in range(len(df_mcu_benchmarking_kem)):
#     # IMPORTANT FIX: check MCU df here, not CPU df
#     if df_mcu_benchmarking_kem.loc[k, "NIST_Category"] == 3:
#         name = df_mcu_benchmarking_kem.loc[k, "Algorithm"]

#         cyc = float(df_mcu_benchmarking_kem.loc[k, "Encapsulation [cycles] (mean)"]) + float(df_mcu_benchmarking_kem.loc[k, "Decapsulation [cycles] (mean)"])
#         b_t = float(df_mcu_benchmarking_kem.loc[k, "PK_bytes"]) + float(df_mcu_benchmarking_kem.loc[k, "CT_bytes"])

#         ram = max(
#             float(df_mcu_benchmarking_kem.loc[k, "Encapsulation [bytes]"]),
#             float(df_mcu_benchmarking_kem.loc[k, "Decapsulation [bytes]"])
#         )

#         mcu_cycles_map[name] = cyc
#         mcu_bytes_map[name]  = b_t
#         mcu_rampeak_map[name] = ram


# # =========================
# # 1) Constants
# # =========================
# STRICT_US = 10_000
# MEDIUM_US = 50_000
# LOOSE_US  = 200_000

# N_pkts_server = 10
# N_pkts_emb    = 6
# N_pkts_cons   = 3

# byte_ref_eth_server = 1500 * N_pkts_server
# byte_ref_mtu_server = 512  * N_pkts_server

# byte_ref_eth_cons = 1500 * N_pkts_cons
# byte_ref_mtu_cons = 512  * N_pkts_cons

# byte_ref_eth_emb  = 1500 * N_pkts_emb
# byte_ref_mtu_emb  = 512  * N_pkts_emb

# # MCU cycle refs (120MHz * time_ref)
# STRICT_CYC  = 1_200_000
# MEDIUM_CYC  = 6_000_000
# LOOSE_CYC   = 24_000_000

# DEFAULT_RAM_REF = 131072  # 128KB, change if you want per-row RAM_ref


# # =========================
# # 2) Utility helpers (ranked list)
# # =========================
# def ranked_server_utility(time_ref_us, bytes_ref, w_cpu=0.7, w_bytes=0.3):
#     out = []
#     for name in cpu_map_server.keys():
#         cpu_norm   = cpu_map_server[name] / time_ref_us
#         bytes_norm = bytes_map_server.get(name, 0.0) / bytes_ref
#         u = w_cpu * cpu_norm + w_bytes * bytes_norm
#         out.append({"Name": name, "utility": u})
#     out.sort(key=lambda x: x["utility"])
#     return out

# def ranked_mcu_utility(cycles_ref, bytes_ref, ram_ref=DEFAULT_RAM_REF, w_cpu=0.6, w_bytes=0.4, w_ram=0.2):
#     # renormalize weights so they sum to 1 (keeps your relative importance)
#     s = w_cpu + w_bytes + w_ram
#     w_cpu, w_bytes, w_ram = w_cpu/s, w_bytes/s, w_ram/s

#     out = []
#     for name in mcu_cycles_map.keys():
#         cpu_norm   = mcu_cycles_map[name] / cycles_ref
#         bytes_norm = mcu_bytes_map.get(name, 0.0) / bytes_ref
#         ram_norm   = mcu_rampeak_map.get(name, 0.0) / ram_ref
#         u = w_cpu * cpu_norm + w_bytes * bytes_norm + w_ram * ram_norm
#         out.append({"Name": name, "utility": u})
#     out.sort(key=lambda x: x["utility"])
#     return out


# # =========================
# # 3) MAIN LOOP: print ONCE per ID, no repeats
# # =========================
# printed_ids = set()

# for i in range(len(df)):  # safer than len(mask), unless mask == df length
#     if df.loc[i, "Possible Deployed CAT"] == "PQC unprotected":
#         continue
#     if df.loc[i, "Has_Enc"] == "NO":
#         continue
#     if df.loc[i, "Possible Deployed CAT"] != 3:
#         continue

#     row_id = df.loc[i, "ID"]
#     if row_id in printed_ids:
#         continue
#     printed_ids.add(row_id)

#     purdue = df.loc[i, "Purdue Layer"]

#     # ---- choose ONE bucket (no triple printing) ----
#     if mask_layer_low[i]:
#         bucket = "Strict"
#         time_ref_us = STRICT_US
#         cycles_ref  = STRICT_CYC
#     elif mask_layer_medium[i]:
#         bucket = "Medium"
#         time_ref_us = MEDIUM_US
#         cycles_ref  = MEDIUM_CYC
#     elif mask_layer_high[i]:
#         bucket = "Loose"
#         time_ref_us = LOOSE_US
#         cycles_ref  = LOOSE_CYC
#     else:
#         bucket = "Medium"
#         time_ref_us = MEDIUM_US
#         cycles_ref  = MEDIUM_CYC

#     # ---- link type ----
#     if mask_protocol_high[i]:
#         link = "Ethernet(1500)"
#         bytes_ref_server = byte_ref_eth_server
#         bytes_ref_cons   = byte_ref_eth_cons
#         bytes_ref_emb    = byte_ref_eth_emb
#     else:
#         link = "lowMTU(512)"
#         bytes_ref_server = byte_ref_mtu_server
#         bytes_ref_cons   = byte_ref_mtu_cons
#         bytes_ref_emb    = byte_ref_mtu_emb

#     # ---- choose class (constrained / server / embedded) ----
#     if mask_3[i] == True:
#         device_class = "constrained"
#         ranked = ranked_mcu_utility(cycles_ref, bytes_ref_cons, ram_ref=DEFAULT_RAM_REF)
#     elif mask_device[i] == True:
#         device_class = "server"
#         ranked = ranked_server_utility(time_ref_us, bytes_ref_server)
#     else:
#         device_class = "embedded"
#         ranked = ranked_mcu_utility(cycles_ref, bytes_ref_emb, ram_ref=DEFAULT_RAM_REF)

#     # ---- PRINT (same style as before) ----
#     print("\n====================================================")
#     print(f"ID: {row_id} | Purdue: {purdue} | class: {device_class} | Bucket: {bucket} | Link: {link}")
#     print("Ranked utility (lower is better):")
#     for r, item in enumerate(ranked, start=1):
#         print(f"{r:02d}. {item['Name']}: {item['utility']:.8f}")


ID: C1 | Purdue: L5–L4 | class: server | Bucket: Loose | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04551819
02. ML-KEM-768: 0.04555085
03. BIKE-L3: 0.18500296

ID: C2 | Purdue: L4–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04575275
02. ML-KEM-768: 0.04588339
03. BIKE-L3: 0.36813184

ID: C4 | Purdue: L3–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04575275
02. ML-KEM-768: 0.04588339
03. BIKE-L3: 0.36813184

ID: C5 | Purdue: L3–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04575275
02. ML-KEM-768: 0.04588339
03. BIKE-L3: 0.36813184

ID: C6 | Purdue: L3–L3 | class: server | Bucket: Medium | Link: Ethernet(1500)
Ranked utility (lower is better):
01. Kyber768: 0.04575275
02. ML-KEM-768: 0.04588339
03. BIKE-L3: 0.36813184

ID: C7 | Purdue: L3–L2 | class: server | Bucket: S

In [576]:
#utility score 

#latency Buckets 

cpu_Total = []
bytes_total = []


for j in range(len(df_cpu_benchmarking_kem["NIST_Category"])):
     if df_cpu_benchmarking_kem["NIST_Category"][j] == 3:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking_kem["decaps__Time_us_mean"][j] + df_cpu_benchmarking_kem["encaps__Time_us_mean"][j] 
         cpu_Total.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking_kem["PK_bytes"][j] + df_cpu_benchmarking_kem["CT_bytes"][j]

         bytes_total.append({
            "Name": df_cpu_benchmarking_kem.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

#us
Strict = 10000 
Medium = 50000 
Loose = 200000 

#allowed packets 

N_pkts_server = 10  #server/gateway 
N_pkts_emb = 6  #Embedded 
N_pkts_cons = 3 #constrained 


byte_ref_etherenet_server = 1500 * N_pkts_server
byte_ref_mtu_server = 512 * N_pkts_server 


# mask_layer = data["Purdue Layer"].astype(str).str.contains(r"\b\b", case = False , na = False)



df["Possible Deployed CAT"]


for i in range(len(mask)):
    if df["Possible Deployed CAT"][i] == "PQC unprotected":
        print("ok")
    else:
        if df["Has_Enc"][i] != "NO":
            if df["Possible Deployed CAT"][i] == 3:
                if mask_layer_low[i]== True:
                    Strict = 10000 
                    print(df["Purdue Layer"][i])
                    print(df["ID"][i])
                    cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total]
                    # print(cpu_Total_scaled)

                    if mask_protocol_high[i] == True:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                        # print(bytes_Total_scaled)
                    else:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                        # print(bytes_Total_scaled)

                    cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                    bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                    w_cpu, w_bytes = 0.7, 0.3

                    utility = []
                    for name in cpu_map.keys():
                        u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                        utility.append({"Name": name, "utility": u})
                    print(utility)


                if mask_layer_medium[i]  == True:
                    Medium = 50000
                    print(df["Purdue Layer"][i])
                    print(df["ID"][i])
                    cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total]
                    # print(cpu_Total_scaled)
                    if mask_protocol_high[i] == True:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                        # print(bytes_Total_scaled)
                    else:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                        # print(bytes_Total_scaled)

                    cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                    bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                    w_cpu, w_bytes = 0.7, 0.3

                    utility = []
                    for name in cpu_map.keys():
                        u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                        utility.append({"Name": name, "utility": u})
                    print(utility)




                if mask_layer_high [i] == True:
                    Loose = 200000 
                    print(df["Purdue Layer"][i])
                    print(df["ID"][i])
                    cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total]
                    # print(cpu_Total_scaled)
                    if mask_protocol_high[i] == True:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                        # print(bytes_Total_scaled)
                    else:
                        bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                        # print(bytes_Total_scaled)

                    cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                    bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                    w_cpu, w_bytes = 0.7, 0.3

                    utility = []
                    for name in cpu_map.keys():
                        u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                        utility.append({"Name": name, "utility": u})
                    print(utility)



# print(bytes_total)
# print(cpu_Total)



L3–L3
C4
[{'Name': 'BIKE-L3', 'utility': 0.368131844}, {'Name': 'Kyber768', 'utility': 0.045752746000000004}, {'Name': 'ML-KEM-768', 'utility': 0.045883394}]
L3–L2
C7
[{'Name': 'BIKE-L3', 'utility': 1.34481922}, {'Name': 'Kyber768', 'utility': 0.04700373}, {'Name': 'ML-KEM-768', 'utility': 0.04765697}]
L3–L2
C7
[{'Name': 'BIKE-L3', 'utility': 0.368131844}, {'Name': 'Kyber768', 'utility': 0.045752746000000004}, {'Name': 'ML-KEM-768', 'utility': 0.045883394}]
ok
L1–L1
C11
[{'Name': 'BIKE-L3', 'utility': 1.34481922}, {'Name': 'Kyber768', 'utility': 0.04700373}, {'Name': 'ML-KEM-768', 'utility': 0.04765697}]
ok
L3–L1
C13
[{'Name': 'BIKE-L3', 'utility': 1.34481922}, {'Name': 'Kyber768', 'utility': 0.04700373}, {'Name': 'ML-KEM-768', 'utility': 0.04765697}]
L3–L1
C13
[{'Name': 'BIKE-L3', 'utility': 0.368131844}, {'Name': 'Kyber768', 'utility': 0.045752746000000004}, {'Name': 'ML-KEM-768', 'utility': 0.045883394}]
L1–L0
C14
[{'Name': 'BIKE-L3', 'utility': 1.5840232824999998}, {'Name': 'Kyber7

In [577]:
# =========================
# 1) Build cpu/bytes maps once (CAT=3 only)
# =========================
cpu_map_raw = {}
bytes_map_raw = {}

for j in range(len(df_cpu_benchmarking_kem)):
    if df_cpu_benchmarking_kem.loc[j, "NIST_Category"] == 3:
        name = df_cpu_benchmarking_kem.loc[j, "Algorithm"]
        cpu_t = float(df_cpu_benchmarking_kem.loc[j, "decaps__Time_us_mean"]) + float(df_cpu_benchmarking_kem.loc[j, "encaps__Time_us_mean"])
        b_t   = float(df_cpu_benchmarking_kem.loc[j, "PK_bytes"]) + float(df_cpu_benchmarking_kem.loc[j, "CT_bytes"])
        cpu_map_raw[name] = cpu_t
        bytes_map_raw[name] = b_t

# =========================
# 2) Constants
# =========================
STRICT_US = 10_000
MEDIUM_US = 50_000
LOOSE_US  = 200_000

N_pkts_server = 10
N_pkts_cons = 3
byte_ref_ethernet_server = 1500 * N_pkts_server
byte_ref_mtu_server      = 512  * N_pkts_cons

w_cpu, w_bytes = 0.7, 0.3

# helper
def ranked_utility(time_ref_us, bytes_ref):
    out = []
    for name in cpu_map_raw.keys():
        cpu_norm   = cpu_map_raw[name] / time_ref_us
        bytes_norm = bytes_map_raw[name] / bytes_ref
        u = w_cpu * cpu_norm + w_bytes * bytes_norm
        out.append({"Name": name, "utility": u})
    out.sort(key=lambda x: x["utility"])
    return out

# =========================
# 3) Loop rows and print ONCE per ID
# =========================
for i in range(len(df)):  # or range(len(mask)) if that's your real row count
    if df.loc[i, "Possible Deployed CAT"] == "PQC unprotected":
        continue
    if df.loc[i, "Has_Enc"] != "NO":
        continue
    if df.loc[i, "Possible Deployed CAT"] != 3:
        continue

    # ---- choose ONLY ONE bucket (prevents repeated prints for same ID) ----
    if mask_layer_low[i]:
        bucket = "Strict"
        time_ref = STRICT_US
    elif mask_layer_medium[i]:
        bucket = "Medium"
        time_ref = MEDIUM_US
    elif mask_layer_high[i]:
        bucket = "Loose"
        time_ref = LOOSE_US
    else:
        # if none matched, skip or default
        # bucket="Medium"; time_ref=MEDIUM_US
        continue

    # ---- choose bytes_ref ----
    if mask_protocol_high[i]:
        link = "Ethernet(1500)"
        bytes_ref = byte_ref_ethernet_server
    else:
        link = "lowMTU(512)"
        bytes_ref = byte_ref_mtu_server

    # ---- compute + print ----
    purdue = df.loc[i, "Purdue Layer"]
    row_id = df.loc[i, "ID"]

    util = ranked_utility(time_ref, bytes_ref)

    print("\n====================================================")
    print(f"ID: {row_id} | Purdue: {purdue} | Bucket: {bucket} ({time_ref} us) | Link: {link} | BYTES_ref={bytes_ref}")
    print("Ranked utility (lower is better):")

    for r, item in enumerate(util, start=1):
        print(f"{r:02d}. {item['Name']}: {item['utility']:.8f}")

In [579]:
df["Has_Enc"]

0                                                   Yes
1                                                   Yes
2                                                   Yes
3                Yes (where IPsec is used), No (legacy)
4                                                   Yes
5                                                   Yes
6                              Yes (when IPsec enabled)
7     Partial (Yes where MACsec/VPN used, No otherwise)
8                                                    No
9                                                   Yes
10                                                  Yes
11               Partial (Yes on Wi-Fi, No on fieldbus)
12                                                  Yes
13                                              Partial
14                                                  Yes
15                                                   No
Name: Has_Enc, dtype: str

In [578]:
df_cpu_benchmarking

,Algorithm,NIST_Category,PK_bytes,SK_bytes,SIG_bytes,keypair__Time_us_mean,sign__Time_us_mean,verify__Time_us_mean,keypair__Time_us_stdev,sign__Time_us_stdev,verify__Time_us_stdev,keypair__HighPrec_ns_mean,sign__HighPrec_ns_mean,verify__HighPrec_ns_mean,keypair__HighPrec_ns_stdev,sign__HighPrec_ns_stdev,verify__HighPrec_ns_stdev,keypair__Iterations,sign__Iterations,verify__Iterations
0,Dilithium2,2.0,1312,2528,2420,21.133,64.181,20.246,29.641,40.509,1.279,21101,64150,20216,29640,40508,1280,141960,46745,148179
1,Dilithium3,3.0,1952,4000,3293,46.921,100.135,31.969,2.423,76.689,9.225,46889,100102,31939,2425,76691,9225,63938,29961,93841
2,Dilithium5,5.0,2592,4864,4595,53.106,119.250,50.323,2.517,61.300,1.361,53076,119213,50292,2518,61301,1357,56491,25158,59615
3,Falcon-1024,5.0,1793,2305,1462,16699.806,308.601,48.131,3973.441,5.258,3.340,16699772,308565,48101,3973438,5259,3340,180,9722,62330
4,Falcon-512,1.0,897,1281,752,5191.730,156.820,24.786,1163.886,5.776,112.612,5191692,156787,24757,1163889,5777,112612,578,19131,121036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,cross-rsdpg-192-fast,3.0,83,48,26772,10.534,790.406,474.641,2.145,6.919,280.571,10504,790361,474605,2142,6915,280561,284787,3796,6321
220,cross-rsdpg-192-small,3.0,83,48,20452,10.532,2037.699,1077.481,2.312,24.596,9.815,10501,2037646,1077437,2312,24605,9819,284854,1473,2785
221,cross-rsdpg-256-balanced,5.0,106,64,40100,17.257,1892.461,999.861,2.282,11.224,6.527,17227,1892404,999816,2282,11229,6527,173844,1586,3001
222,cross-rsdpg-256-fast,5.0,106,64,48102,17.269,1477.215,871.743,2.309,47.237,10.625,17239,1477154,871698,2308,47242,10624,173724,2031,3442


In [580]:
df.loc[12, "Auth_Type"] = "ServerCert"
df.loc[12, "Auth_Type"]

'ServerCert'

In [581]:
df.loc[13, "Auth_Type"] = "MutCert"
df.loc[13, "Auth_Type"]

'MutCert'

In [582]:
df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,Data Lifetime,Contains PII?,Exposure,Device,Firmware upgradable,Crypto modifiable,Vendor Support of PQC,Possible Deployed CAT,Enc_Ranking_By_Utility,Enc_Best_Alg
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,5–10y,No,Internet / external B2B (High),"Market systems, EMS/market servers",Y,Y,N/Unknown,5,"Kyber1024:0.06282556, ML-KEM-1024:0.06288374, ...",Kyber1024
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,~5–10y,No,Utility WAN / extranet (Medium),EMS / DMS application servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,10y+,No / Possible,Utility WAN (Medium),MDMS servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,"Telemetry days, logs 5+ years",No,Utility WAN (Medium),"SCADA master, automation headend servers",Y,Y,N/Unknown,3,"Kyber768:0.04575275, ML-KEM-768:0.04588339, BI...",Kyber768
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,~5–10y,No,Utility WAN (Medium),DMS & automation headend servers,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,7–10y,Yes,Utility WAN / data centre network (Medium),"AMI headend, MDMS servers",Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,"Years (event logs, switching records)",No,Long-distance WAN / wireless (High),Automation frontend (RTU/IED gateway),Maybe,Maybe,N,3,"Kyber768:0.04700373, ML-KEM-768:0.04765697, BI...",Kyber768
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,Logs 5+ years,No,Substation LAN (Low–Medium),Substation controller / bay controllers (Resou...,Maybe,Maybe,N,1,"ml-kem-512:0.48480465, bikel1:72.16349553, hqc...",ml-kem-512
8,C9,Primary Substation Node ↔ Local I/O,Primary Substation – Process,L1–L0,"Modbus RS485, IEC 61850 Ethernet, analog loops",None,No,NaN,No,None (physical isolation only),...,"Real-time, logs <1y",No,Local wired network (Low),"IEDs, sensors, actuators (Resource Constrained)",Limited/Maybe,No,N,PQC unprotected,,
9,C10,Secondary Substation Node / Concentrator ↔ Sma...,Secondary Substation (med/low volt.) – Custome...,L2–L1,"G3-PLC, PRIME, CX1, LonTalk over PLC; DLMS/COS...",DLMS/COSEM security (AES),Yes,PSK / shared symmetric keys (per meter or per ...,Yes (DLMS MAC),EndToEnd (between meter and headend/concentrator),...,7–10y,Yes,Field PLC over public LV grid (High),"Smart meters (Resource Constrained)

In [583]:
mask_auth = df["Auth_Type"].astype(str).str.contains(r"\bServerCert\b", case=False, na=False)
mask_auth_mut = df["Auth_Type"].astype(str).str.contains(r"\bMutCert\b", case=False, na=False)

mask_auth_mut

0     False
1     False
2     False
3      True
4      True
5      True
6      True
7     False
8     False
9     False
10    False
11    False
12    False
13     True
14    False
15    False
Name: Auth_Type, dtype: bool

In [584]:
cols = [
    "Key Gen [Cylces](mean)", "Key Gen [Cylces](min)", "Key Gen [Cylces](max)",
    "Sign[Cycles](mean)", "Sign[Cycles](max)", "Sign[Cycles](min)",
    "Verify[Cycles](mean)", "Verify[Cycles](max)", "Verify[Cycles](min)"
]
df_mcu_benchmarking[cols] = df_mcu_benchmarking[cols].fillna(0)

df_mcu_benchmarking.info()

<class 'pandas.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Algorithm               27 non-null     str    
 1   NIST_Category           21 non-null     float64
 2   PK_bytes                27 non-null     int64  
 3   SK_bytes                27 non-null     int64  
 4   SIG_bytes               27 non-null     int64  
 5   Executions              0 non-null      float64
 6   Implementation          27 non-null     str    
 7   .text [bytes]           27 non-null     int64  
 8   .data [bytes]           27 non-null     int64  
 9   .bss [bytes]            27 non-null     int64  
 10  Total [bytes]           27 non-null     int64  
 11  Key Generation [bytes]  21 non-null     float64
 12  Sign(bytes)             21 non-null     float64
 13  Verify(bytes)           21 non-null     float64
 14  Implementations         21 non-null     str    
 15  Ke

In [585]:
#utility score 
#Authentication utility score experiemental one 


#latency Buckets 

cpu_Total_sig = []
bytes_total_sig = []

cpu_Total_1_sig = []
bytes_total_1_sig =[]

cpu_Total_5_sig = []
bytes_total_5_sig =[]

for j in range(len(df_cpu_benchmarking["NIST_Category"])):
     if df_cpu_benchmarking["NIST_Category"][j] == 3:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking["sign__Time_us_mean"][j] + df_cpu_benchmarking["verify__Time_us_mean"][j] 
         cpu_Total_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking["PK_bytes"][j] + df_cpu_benchmarking["SIG_bytes"][j]

         bytes_total_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

     if df_cpu_benchmarking["NIST_Category"][j] == 1:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking["sign__Time_us_mean"][j] + df_cpu_benchmarking["verify__Time_us_mean"][j] 
         cpu_Total_1_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking["PK_bytes"][j] + df_cpu_benchmarking["SIG_bytes"][j]

         bytes_total_1_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })
    
     if df_cpu_benchmarking["NIST_Category"][j] == 5:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_cpu_benchmarking["sign__Time_us_mean"][j] + df_cpu_benchmarking["verify__Time_us_mean"][j] 
         cpu_Total_5_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_cpu_benchmarking["PK_bytes"][j] + df_cpu_benchmarking["SIG_bytes"][j]

         bytes_total_5_sig.append({
            "Name": df_cpu_benchmarking.loc[j, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })



mcu_total_sig = []
bytes_total_mcu_sig = []
ram_peak_sig = []

mcu_total_1_sig = []
bytes_total_mcu_1_sig = []
ram_peak_1_sig = []

mcu_total_5_sig = []
bytes_total_mcu_5_sig = []
ram_peak_5_sig = []

for k in range(len(df_mcu_benchmarking_kem["NIST_Category"])):
     if df_mcu_benchmarking["NIST_Category"][k] == 3:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking_kem["Sign[Cycles](mean)"][k] + df_mcu_benchmarking_kem["Verify[Cycles](min)"][k] 
         mcu_total_sig.append({
            "Name": df_mcu_benchmarking_kem.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking["PK_bytes"][k] + df_mcu_benchmarking_kem["SIG_bytes"][k]

         bytes_total_mcu_sig.append({
            "Name": df_mcu_benchmarking.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking["Sign(bytes)"][k], df_mcu_benchmarking_kem["Verify(bytes)"][k])

         ram_peak_sig.append({
             "Name":df_mcu_benchmarking.loc[k, "Algorithm"],
             "value": ram
         })
    
     if df_mcu_benchmarking["NIST_Category"][k] == 1:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking["Sign[Cycles](mean)"][k] + df_mcu_benchmarking["Verify[Cycles](min)"][k] 
         mcu_total_1_sig.append({
            "Name": df_mcu_benchmarking.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking["PK_bytes"][k] + df_mcu_benchmarking["SIG_bytes"][k]

         bytes_total_mcu_1_sig.append({
            "Name": df_mcu_benchmarking.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking["Sign(bytes)"][k], df_mcu_benchmarking["Verify(bytes)"][k])

         ram_peak_1_sig.append({
             "Name":df_mcu_benchmarking.loc[k, "Algorithm"],
             "value": ram
         })
        
     if df_mcu_benchmarking["NIST_Category"][k] == 5:
        #  print(df_cpu_benchmarking_kem["NIST_Category"][j]) 
         cpu_T = df_mcu_benchmarking["Sign[Cycles](mean)"][k] + df_mcu_benchmarking["Verify[Cycles](min)"][k] 
         mcu_total_5_sig.append({
            "Name": df_mcu_benchmarking.loc[k, "Algorithm"],  # change if your column name differs
            "value": cpu_T
         })

         bytes_t = df_mcu_benchmarking["PK_bytes"][k] + df_mcu_benchmarking["SIG_bytes"][k]

         bytes_total_mcu_5_sig.append({
            "Name": df_mcu_benchmarking.loc[k, "Algorithm"],  # change if your column name differs
            "value": bytes_t
         })

         ram = max(df_mcu_benchmarking["Sign(bytes)"][k], df_mcu_benchmarking["Verify(bytes)"][k])

         ram_peak_5_sig.append({
             "Name":df_mcu_benchmarking.loc[k, "Algorithm"],
             "value": ram
         })


#us
Strict = 10000 
Medium = 50000 
Loose = 200000 


#allowed packets 

N_pkts_server = 10  #server/gateway 
N_pkts_emb = 6  #Embedded 
N_pkts_cons = 3 #constrained 


byte_ref_etherenet_server = 1500 * N_pkts_server
byte_ref_mtu_server = 512 * N_pkts_server 

byte_ref_etherenet_cons = 1500 * N_pkts_cons
byte_ref_mtu_cons = 512 * N_pkts_cons

byte_ref_etherenet_emb = 1500 * N_pkts_emb
byte_ref_mtu_emb = 512 * N_pkts_emb

f = 120 #MHZ
Strict_mcu = 1200000
Medium_mcu = 6000000
Loose_MCU = 24000000


for i in range(len(mask)):
    if df["Possible Deployed CAT"][i] == "PQC unprotected":
        print("ok")
    else:
        if mask_3[i] == True: #checking resource constrained
            if mask_auth[i] == True: #servercert 
                if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #nist catagory - 1
                if df["Possible Deployed CAT"][i] == 1:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #NIST catagory  -5         

                if df["Possible Deployed CAT"][i] == 5:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)
                        

            if mask_auth_mut[i] == True: #servercert 
                if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]* 2) / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_cons} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 1:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_cons} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 5:
                     
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_cons} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.2 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

        elif mask_device[i] == True:
             if mask_auth[i] == True:
                if df["Possible Deployed CAT"][i] == 3:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #nist catagory - 1 

                if df["Possible Deployed CAT"][i] == 1:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                #NIST catagory - 5         
                if df["Possible Deployed CAT"][i] == 5:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Medium} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Loose} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

             if mask_auth_mut[i] == True:
                if df["Possible Deployed CAT"][i] == 3:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Strict} for x in cpu_Total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Medium} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Loose} for x in cpu_Total]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 1:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Strict} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Medium} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Loose} for x in cpu_Total_1]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_1]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 5:
                    if mask_layer_low[i]== True:
                        Strict = 10000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Strict} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_medium[i]  == True:
                        Medium = 50000
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Medium} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                    if mask_layer_high [i] == True:
                        Loose = 200000 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        cpu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / Loose} for x in cpu_Total_5]
                        # print(cpu_Total_scaled)
                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_etherenet_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]*2) / byte_ref_mtu_server} for x in bytes_total_5]
                            # print(bytes_Total_scaled)

                        cpu_map   = {d["Name"]: float(d["value"]) for d in cpu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled}

                        w_cpu, w_bytes = 0.7, 0.3

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

        else: #checking embeded
            if mask_auth[i] == True:
                if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 1:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                if df["Possible Deployed CAT"][i] == 5:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"]) / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_etherenet_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"]) / byte_ref_mtu_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

            if mask_auth_mut[i] == True:

                 if df["Possible Deployed CAT"][i] == 3:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"])*2 / Strict_mcu} for x in mcu_total]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_etherenet_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_mtu_emb} for x in bytes_total_mcu]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                 if df["Possible Deployed CAT"][i] == 1:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"])*2 / Strict_mcu} for x in mcu_total_1]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_etherenet_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_mtu_emb} for x in bytes_total_mcu_1]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_1]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)

                 if df["Possible Deployed CAT"][i] == 5:
                    # print("details coming up")
                    if mask_layer_low[i]== True: 
                        print(df["Purdue Layer"][i])
                        print(df["ID"][i])
                        mcu_Total_scaled = [{"Name": x["Name"], "value": float(x["value"])*2 / Strict_mcu} for x in mcu_total_5]
                        # print(cpu_Total_scaled)

                        if mask_protocol_high[i] == True:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_etherenet_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        else:
                            bytes_Total_scaled_mcu = [{"Name": x["Name"], "value": float(x["value"])*2 / byte_ref_mtu_emb} for x in bytes_total_mcu_5]
                            # print(bytes_Total_scaled)
                        
                        ram_peak_mcu =  [{"Name": x["Name"], "value": float(x["value"])} for x in ram_peak_5]

                        cpu_map   = {d["Name"]: float(d["value"]) for d in mcu_Total_scaled}
                        bytes_map = {d["Name"]: float(d["value"]) for d in bytes_Total_scaled_mcu}
                        ram_map = {d["Name"]: float(d["value"]) for d in ram_peak_mcu}

                        w_cpu, w_bytes , w_ram = 0.6, 0.4 , 0.2

                        utility = []
                        for name in cpu_map.keys():
                            u = w_cpu * cpu_map.get(name, 0.0) + w_bytes * bytes_map.get(name, 0.0) + w_ram * ram_map.get(name, 0.0)
                            utility.append({"Name": name, "utility": u})
                        print(utility)


                    

                

KeyError: 'Sign[Cycles](mean)'

In [602]:
import pandas as pd

# ============================================================
# AUTHENTICATION utility (Signatures) - CLEAN version
# CAT 1/3/5 + server/constrained/embedded + ServerCert vs MutCert
# ============================================================

# ---------- helpers ----------
def safe_int_cat(x):
    if isinstance(x, (int, float)) and x in (1, 3, 5):
        return int(x)
    if isinstance(x, str):
        x = x.strip()
        if x.isdigit():
            v = int(x)
            if v in (1, 3, 5):
                return v
    return None

def find_col(df, candidates):
    """Find a column robustly (exact, stripped, or substring match)."""
    cols = list(df.columns)
    strip_map = {c.strip(): c for c in cols}

    # exact
    for cand in candidates:
        if cand in cols:
            return cand
    # exact after strip
    for cand in candidates:
        if cand in strip_map:
            return strip_map[cand]
    # substring
    for c in cols:
        lc = c.lower()
        for cand in candidates:
            if cand.lower() in lc:
                return c
    raise KeyError(f"None of candidates found: {candidates}")

def normalize_weights(*ws):
    s = sum(ws)
    if s == 0:
        return ws
    return tuple(w/s for w in ws)

# ---------- build maps once ----------
def build_server_sig_maps(df_cpu_sig, cats=(1,3,5)):
    # maps[cat] = {"cpu_us": {alg: us}, "bytes": {alg: bytes}}
    maps = {c: {"cpu_us": {}, "bytes": {}} for c in cats}

    for j in range(len(df_cpu_sig)):
        cat_raw = df_cpu_sig.loc[j, "NIST_Category"]
        if pd.isna(cat_raw):
            continue
        try:
            cat = int(float(cat_raw))
        except Exception:
            continue
        if cat not in maps:
            continue

        alg = df_cpu_sig.loc[j, "Algorithm"]

        # cpu = sign + verify (mean)
        cpu_us = float(df_cpu_sig.loc[j, "sign__Time_us_mean"]) + float(df_cpu_sig.loc[j, "verify__Time_us_mean"])

        # bytes = pk + sig
        b = float(df_cpu_sig.loc[j, "PK_bytes"]) + float(df_cpu_sig.loc[j, "SIG_bytes"])

        maps[cat]["cpu_us"][alg] = cpu_us
        maps[cat]["bytes"][alg]  = b

    return maps

def build_mcu_sig_maps(df_mcu_sig, cats=(1,3,5)):
    # maps[cat] = {"cycles": {alg: cycles}, "bytes": {alg: bytes}, "ram": {alg: ram_peak_bytes}}
    maps = {c: {"cycles": {}, "bytes": {}, "ram": {}} for c in cats}

    # Robust column lookup (because your MCU sheet column names can vary)
    col_sign_cyc = find_col(df_mcu_sig, ["Sign[Cycles](mean)", "Sign[Cycles] (mean)", "Sign Cycles (mean)", "Sign[Cycles]"])
    col_ver_cyc  = find_col(df_mcu_sig, ["Verify[Cycles](mean)", "Verify[Cycles] (mean)", "Verify Cycles (mean)", "Verify[Cycles]"])
    col_sign_ram = find_col(df_mcu_sig, ["Sign(bytes)", "Sign [bytes]", "Sign (bytes)"])
    col_ver_ram  = find_col(df_mcu_sig, ["Verify(bytes)", "Verify [bytes]", "Verify (bytes)"])

    for k in range(len(df_mcu_sig)):
        cat_raw = df_mcu_sig.loc[k, "NIST_Category"]
        if pd.isna(cat_raw):
            continue
        try:
            cat = int(float(cat_raw))
        except Exception:
            continue
        if cat not in maps:
            continue

        alg = df_mcu_sig.loc[k, "Algorithm"]

        # cycles = sign(mean) + verify(mean)
        cyc = float(df_mcu_sig.loc[k, col_sign_cyc]) + float(df_mcu_sig.loc[k, col_ver_cyc])

        # bytes = pk + sig (these must exist in df_mcu_sig)
        b = float(df_mcu_sig.loc[k, "PK_bytes"]) + float(df_mcu_sig.loc[k, "SIG_bytes"])

        # RAM_peak = max(sign_bytes, verify_bytes)
        ram_peak = max(float(df_mcu_sig.loc[k, col_sign_ram]), float(df_mcu_sig.loc[k, col_ver_ram]))

        maps[cat]["cycles"][alg] = cyc
        maps[cat]["bytes"][alg]  = b
        maps[cat]["ram"][alg]    = ram_peak

    return maps

def rank_server_sig(server_map, time_ref_us, bytes_ref, mult=1, w_cpu=0.7, w_bytes=0.3):
    cpu_map = server_map["cpu_us"]
    b_map   = server_map["bytes"]
    out = []
    for alg in cpu_map.keys():
        cpu_norm = (cpu_map[alg] * mult) / time_ref_us
        b_norm   = (b_map.get(alg, 0.0) * mult) / bytes_ref
        u = w_cpu*cpu_norm + w_bytes*b_norm
        out.append({"Name": alg, "utility": u})
    out.sort(key=lambda x: x["utility"])
    return out

def rank_mcu_sig(mcu_map, cycles_ref, bytes_ref, ram_ref, mult=1, w_cpu=0.6, w_bytes=0.2, w_ram=0.2):
    # For mutual, cycles/bytes scale by 2; RAM_peak typically does NOT double → keep RAM as-is
    w_cpu, w_bytes, w_ram = normalize_weights(w_cpu, w_bytes, w_ram)

    c_map = mcu_map["cycles"]
    b_map = mcu_map["bytes"]
    r_map = mcu_map["ram"]
    out = []
    for alg in c_map.keys():
        cpu_norm = (c_map[alg] * mult) / cycles_ref
        b_norm   = (b_map.get(alg, 0.0) * mult) / bytes_ref
        r_norm   = (r_map.get(alg, 0.0)) / ram_ref
        u = w_cpu*cpu_norm + w_bytes*b_norm + w_ram*r_norm
        out.append({"Name": alg, "utility": u})
    out.sort(key=lambda x: x["utility"])
    return out


# ---------- build maps ----------
server_sig_maps = build_server_sig_maps(df_cpu_benchmarking, cats=(1,3,5))
mcu_sig_maps    = build_mcu_sig_maps(df_mcu_benchmarking, cats=(1,3,5))


# ---------- refs/constants ----------
STRICT_US, MEDIUM_US, LOOSE_US = 10_000, 50_000, 200_000
STRICT_CYC, MEDIUM_CYC, LOOSE_CYC = 1_200_000, 6_000_000, 24_000_000  # 120MHz * {10,50,200}ms

N_pkts_server, N_pkts_emb, N_pkts_cons = 10, 6, 3

byte_ref_eth_server = 1500 * N_pkts_server
byte_ref_mtu_server = 512  * N_pkts_server

byte_ref_eth_emb = 1500 * N_pkts_emb
byte_ref_mtu_emb = 512  * N_pkts_emb

byte_ref_eth_cons = 1500 * N_pkts_cons
byte_ref_mtu_cons = 512  * N_pkts_cons

DEFAULT_RAM_REF = 131072  # 128KB (change if you have per-row RAM budgets)


# ---------- output columns ----------
df["Auth_Ranking_By_Utility"] = ""
df["Auth_Best_Alg"] = ""
df["Auth_Best_Utility"] = 0.0


# ---------- main loop (print once per ID, store back) ----------
printed_ids = set()

for i in range(len(df)):
    if df.loc[i, "Possible Deployed CAT"] == "PQC unprotected":
        continue

    cat = safe_int_cat(df.loc[i, "Possible Deployed CAT"])
    if cat is None:
        continue

    row_id = df.loc[i, "ID"]
    if row_id in printed_ids:
        continue
    printed_ids.add(row_id)

    purdue = df.loc[i, "Purdue Layer"]

    # Skip if NOT cert-based auth (PSK etc.)
    # mask_auth = ServerCert, mask_auth_mut = MutCert
    if not (mask_auth[i] or mask_auth_mut[i]):
        continue

    # ServerCert vs MutCert multiplier
    mult = 2 if mask_auth_mut[i] else 1
    auth_mode = "MutCert" if mult == 2 else "ServerCert"

    # pick ONE bucket
    if mask_layer_low[i]:
        bucket = "Strict"
        time_ref_us = STRICT_US
        cycles_ref  = STRICT_CYC
    elif mask_layer_medium[i]:
        bucket = "Medium"
        time_ref_us = MEDIUM_US
        cycles_ref  = MEDIUM_CYC
    elif mask_layer_high[i]:
        bucket = "Loose"
        time_ref_us = LOOSE_US
        cycles_ref  = LOOSE_CYC
    else:
        bucket = "Medium"
        time_ref_us = MEDIUM_US
        cycles_ref  = MEDIUM_CYC

    # link type
    if mask_protocol_high[i]:
        link = "Ethernet(1500)"
        bytes_ref_server = byte_ref_eth_server
        bytes_ref_emb    = byte_ref_eth_emb
        bytes_ref_cons   = byte_ref_eth_cons
    else:
        link = "lowMTU(512)"
        bytes_ref_server = byte_ref_mtu_server
        bytes_ref_emb    = byte_ref_mtu_emb
        bytes_ref_cons   = byte_ref_mtu_cons

    # device class
    if mask_3[i] == True:
        device_class = "constrained"
        ranked = rank_mcu_sig(mcu_sig_maps[cat], cycles_ref, bytes_ref_cons, ram_ref=DEFAULT_RAM_REF,
                              mult=mult, w_cpu=0.6, w_bytes=0.2, w_ram=0.2)
    elif mask_device[i] == True:
        device_class = "server"
        ranked = rank_server_sig(server_sig_maps[cat], time_ref_us, bytes_ref_server,
                                 mult=mult, w_cpu=0.7, w_bytes=0.3)
    else:
        device_class = "embedded"
        ranked = rank_mcu_sig(mcu_sig_maps[cat], cycles_ref, bytes_ref_emb, ram_ref=DEFAULT_RAM_REF,
                              mult=mult, w_cpu=0.6, w_bytes=0.2, w_ram=0.2)

    if len(ranked) == 0:
        continue

    # print
    print("\n====================================================")
    print(f"ID: {row_id} | CAT:{cat} | Purdue:{purdue} | class:{device_class} | bucket:{bucket} | link:{link} | auth:{auth_mode} (x{mult})")
    print("Ranked utility (lower is better):")
    for r, item in enumerate(ranked, start=1):
        print(f"{r:02d}. {item['Name']}: {item['utility']:.8f}")

    # store back to df for all rows with same ID
    ranking_str = ", ".join([f'{x["Name"]}:{x["utility"]:.8f}' for x in ranked])
    best = ranked[0]

    df.loc[df["ID"] == row_id, "Auth_Ranking_By_Utility"] = ranking_str
    df.loc[df["ID"] == row_id, "Auth_Best_Alg"] = best["Name"]
    df.loc[df["ID"] == row_id, "Auth_Best_Utility"] = best["utility"]


ID: C1 | CAT:5 | Purdue:L5–L4 | class:server | bucket:Loose | link:Ethernet(1500) | auth:ServerCert (x1)
Ranked utility (lower is better):
01. Falcon-padded-1024: 0.10332705
02. Falcon-1024: 0.10939183
03. Dilithium5: 0.23999060
04. ML-DSA-87: 0.24202067
05. cross-rsdpg-256-small: 1.23126212
06. cross-rsdpg-256-balanced: 1.34743081
07. cross-rsdpg-256-fast: 1.61280573
08. cross-rsdp-256-small: 1.71770611
09. cross-rsdp-256-balanced: 1.80145928
10. SPHINCS+-SHA2-256f-simple: 1.81582014
11. SLH_DSA_SHA2_512_PREHASH_SHA2_256F: 1.84459674
12. SLH_DSA_SHA2_224_PREHASH_SHA2_256F: 1.84466474
13. SLH_DSA_SHA2_512_256_PREHASH_SHA2_256F: 1.84468039
14. SLH_DSA_SHA3_512_PREHASH_SHA2_256F: 1.84469844
15. SLH_DSA_SHA2_512_224_PREHASH_SHA2_256F: 1.84471749
16. SLH_DSA_PURE_SHA2_256F: 1.84476661
17. SLH_DSA_SHA3_384_PREHASH_SHA2_256F: 1.84516873
18. SLH_DSA_SHA2_256_PREHASH_SHA2_256F: 1.84559123
19. SLH_DSA_SHA2_384_PREHASH_SHA2_256F: 1.84941259
20. SLH_DSA_SHA3_256_PREHASH_SHA2_256F: 1.84956749
21.

In [603]:
dcols_to_drop = ["Auth_Best_Utility"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,Vendor Support of PQC,Possible Deployed CAT,Enc_Ranking_By_Utility,Enc_Best_Alg,Auth_Ranking_By_Utility,Auth_Best_Alg,Auth_Best_Utility,Enc_Confidence_Top3,Auth_Confidence_Top3,Migration_Priority
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,N/Unknown,5,"Kyber1024:0.06282556, ML-KEM-1024:0.06288374, ...",Kyber1024,"Falcon-padded-1024:0.10332705, Falcon-1024:0.1...",Falcon-padded-1024,0.103327,"Kyber1024:0.523135, ML-KEM-1024:0.476865, BIKE...","Falcon-padded-1024:0.996982, Falcon-1024:0.003...",0.624 (High)
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.10600819, Falcon-1024:0.1...",Falcon-padded-1024,0.106008,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.10600819, Falcon-1024:0.1...",Falcon-padded-1024,0.106008,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,N/Unknown,3,"Kyber768:0.04575275, ML-KEM-768:0.04588339, BI...",Kyber768,"Dilithium3:0.35230875, ML-DSA-65:0.35941413, c...",Dilithium3,0.352309,"Kyber768:0.570903, ML-KEM-768:0.429097, BIKE-L...","Dilithium3:0.986069, ML-DSA-65:0.013931, cross...",0.521 (High)
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.21201639, Falcon-1024:0.2...",Falcon-padded-1024,0.212016,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.571 (High)
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.21201639, Falcon-1024:0.2...",Falcon-padded-1024,0.212016,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,N,3,"Kyber768:0.04700373, ML-KEM-768:0.04765697, BI...",Kyber768,"Dilithium3:0.36287707, ML-DSA-65:0.39413733, c...",Dilithium3,0.362877,"Kyber768:0.800554, ML-KEM-768:0.199446, BIKE-L...","Dilithium3:1.000000, ML-DSA-65:0.000000, cross...",0.352 (Medium)
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,N,1,"ml-kem-512:0.48480465, bikel1:72.16349553, hqc...",ml-kem-512,,,0.000000,"ml-k

Confidence Score 

In [591]:
df["Enc_Ranking_By_Utility"][0]



'Kyber1024:0.06282556, ML-KEM-1024:0.06288374, BIKE-L5:0.35523461'

In [592]:
import pandas as pd
import numpy as np
import re

COL = "Enc_Ranking_By_Utility"
NEWCOL_TOP3 = "Enc_Confidence_Top3"
NEWCOL_TOP1 = "Enc_Confidence_Top1"

_float_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def parse_alg_utils(cell):
    if pd.isna(cell):
        return [], []
    s = str(cell).replace("…", "").replace("...", "").strip()
    if not s:
        return [], []

    algs, utils = [], []
    for part in s.split(","):
        part = part.strip()
        if ":" not in part:
            continue
        alg, val = part.split(":", 1)
        alg = alg.strip()
        val = val.strip()

        # robust float
        try:
            u = float(val)
        except ValueError:
            m = _float_re.search(val)
            if not m:
                continue
            u = float(m.group(0))

        algs.append(alg)
        utils.append(u)

    return algs, utils

def softmax_probs_from_util(cell, alpha=0.01, eps=1e-12):
    algs, utils = parse_alg_utils(cell)
    if not algs:
        return [], np.array([])

    u = np.array(utils, dtype=float)
    umin = float(u.min())
    tau = max(alpha * umin, eps)

    s = -(u - umin) / tau
    s = s - s.max()             # stable
    e = np.exp(s)
    p = e / e.sum()
    return algs, p

def topk_confidence_string(cell, k=3, alpha=0.01, digits=6):
    algs, p = softmax_probs_from_util(cell, alpha=alpha)
    if len(algs) == 0:
        return np.nan

    items = sorted(zip(algs, p), key=lambda x: x[1], reverse=True)[:k]
    return ", ".join([f"{a}:{v:.{digits}f}" for a, v in items])

def top1_confidence(cell, alpha=0.01):
    algs, p = softmax_probs_from_util(cell, alpha=alpha)
    if p.size == 0:
        return np.nan
    return float(p.max())

# ✅ add columns safely
df[NEWCOL_TOP3] = df[COL].apply(lambda x: topk_confidence_string(x, k=3, alpha=0.01))
df[NEWCOL_TOP1] = df[COL].apply(lambda x: top1_confidence(x, alpha=0.01))

In [593]:
df.drop(columns=["Enc_Confidence_Top1"], inplace=True, errors="ignore")

df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,Firmware upgradable,Crypto modifiable,Vendor Support of PQC,Possible Deployed CAT,Enc_Ranking_By_Utility,Enc_Best_Alg,Auth_Ranking_By_Utility,Auth_Best_Alg,Auth_Best_Utility,Enc_Confidence_Top3
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,Y,Y,N/Unknown,5,"Kyber1024:0.06282556, ML-KEM-1024:0.06288374, ...",Kyber1024,"Falcon-padded-1024:0.06271120, Falcon-1024:0.0...",Falcon-padded-1024,0.062711,"Kyber1024:0.523135, ML-KEM-1024:0.476865, BIKE..."
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.06646480, Falcon-1024:0.0...",Falcon-padded-1024,0.066465,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE..."
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.06646480, Falcon-1024:0.0...",Falcon-padded-1024,0.066465,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE..."
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,Y,Y,N/Unknown,3,"Kyber768:0.04575275, ML-KEM-768:0.04588339, BI...",Kyber768,"Dilithium3:0.21349891, ML-DSA-65:0.22259312, c...",Dilithium3,0.213499,"Kyber768:0.570903, ML-KEM-768:0.429097, BIKE-L..."
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.13292961, Falcon-1024:0.1...",Falcon-padded-1024,0.132930,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE..."
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,Y,Y,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.13292961, Falcon-1024:0.1...",Falcon-padded-1024,0.132930,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE..."
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,Maybe,Maybe,N,3,"Kyber768:0.04700373, ML-KEM-768:0.04765697, BI...",Kyber768,"Dilithium3:0.22829456, ML-DSA-65:0.27120560, c...",Dilithium3,0.228295,"Kyber768:0.800554, ML-KEM-768:0.199446, BIKE-L..."
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,Maybe,Maybe,N,1,"ml-kem-512:0.48480465, bikel1:72.16349553, hqc...",ml-kem-512,,,0.000000,"ml-kem-512:1.000000, bikel1:0.000000, hqc-128:..."
8,C9,Primary Substation Node ↔ Local I/O,Primary Substation – Process,L1–L0,"Modbus RS485, IEC 61850 Ethernet, analog loops",None,No,NaN,No,None (physical isolation only),...,Limited/Maybe,No,N,PQC unprotected,,,,,0.000000,NaN
9,C10,Secondary Substation Node / Concentrator ↔ Sma...,Secondary Substation (med/low volt.) – Custome...,L2–L1,"G3-PLC, PRIME, CX1, LonT

In [594]:
df["Auth_Confidence_Top3"] = df["Auth_Ranking_By_Utility"].apply(
    lambda x: topk_confidence_string(x, k=3, alpha=0.01, digits=6)
)

In [595]:
df["Auth_Confidence_Top3"][0]

'Falcon-padded-1024:0.996982, Falcon-1024:0.003018, Dilithium5:0.000000'

Migration priority 

In [596]:
import re
import numpy as np
import pandas as pd

# -------------------------
# helpers: column finding + text mapping
# -------------------------
def find_col(df, *candidates):
    """Find a column by case-insensitive match ignoring spaces/underscores."""
    norm = {re.sub(r"[\s_]+", "", c).lower(): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r"[\s_]+", "", cand).lower()
        if key in norm:
            return norm[key]
    return None

def as_text(x):
    if pd.isna(x): 
        return ""
    return str(x).strip()

def yn_maybe(x, unknown=0.5):
    s = as_text(x).lower()
    if s in {"y", "yes", "true", "1"}: return 1.0
    if s in {"n", "no", "false", "0"}: return 0.0
    if "maybe" in s or "partial" in s: return 0.5
    if "unknown" in s or s == "": return unknown
    return unknown

# -------------------------
# A) Risk urgency R = 0.45E + 0.35I + 0.20L  (optional PII)
# -------------------------
def exposure_score_from_protocols(proto):
    s = as_text(proto).lower()
    if "internet" in s or "external" in s or "b2b" in s:
        return 1.0
    if "wan" in s or "extranet" in s or "cell" in s or "phone" in s:
        return 0.7
    if "lan" in s or "ethernet" in s or "internal" in s or "intranet" in s:
        return 0.4
    if "isolated" in s or "offline" in s:
        return 0.1
    return 0.4  # default

def business_impact_score(x):
    s = as_text(x).lower()
    if "high" in s: return 1.0
    if "medium" in s: return 0.6
    if "low" in s: return 0.3
    return np.nan

def lifetime_score(x):
    s = as_text(x).lower().replace(" ", "")
    if not s: return np.nan
    if "10y" in s or "10+" in s: return 1.0
    if "5-10" in s or "5to10" in s: return 0.8
    if "1-5" in s or "1to5" in s: return 0.5
    if "<1" in s or "0-1" in s or "lessthan1" in s: return 0.2
    return np.nan

def purdue_layer_to_impact(purdue):
    """Fallback: lower layers more impactful (rough proxy)."""
    s = as_text(purdue).upper()
    nums = [int(n) for n in re.findall(r"\d", s)]
    if not nums: 
        return 0.6
    m = min(nums)  # lower layer number = more critical
    if m <= 2: return 1.0
    if m == 3: return 0.7
    return 0.3

# -------------------------
# B) Feasibility F = 0.4FW + 0.4CM + 0.2V
# -------------------------

# -------------------------
# C) Cost/Friction C = 0.4S + 0.3Pu + 0.3D
# -------------------------
def placement_friction(sec_place):
    s = as_text(sec_place).lower()
    if "endtoend" in s or "end-to-end" in s: return 1.0
    if "gatewayonly" in s or "gateway-only" in s: return 0.6
    if "vpn" in s or "partial" in s: return 0.4
    return 0.6  # default-ish

def purdue_friction(purdue):
    """
    Slide: L0–L2=1.0, L3=0.7, L4–L5=0.4
    If value contains multiple layers (e.g., L4-L3), take MAX friction (hardest).
    """
    s = as_text(purdue).upper()
    nums = [int(n) for n in re.findall(r"\d", s)]
    if not nums:
        return 0.7
    def fr(layer):
        if layer <= 2: return 1.0
        if layer == 3: return 0.7
        return 0.4
    return max(fr(n) for n in nums)

def device_friction(devtype):
    s = as_text(devtype).lower()
    if "constrained" in s: return 1.0
    if "embedded" in s or "mcu" in s: return 0.7
    if "server" in s or "gateway" in s: return 0.3
    return 0.7  # default

# -------------------------
# MAIN: compute + PRINT (no new columns)
# -------------------------
# Try to locate your columns (adjust candidates if needed)
col_protocols = find_col(df, "Protocols", "Protocol")
col_purdue    = find_col(df, "Purdue Layer", "Purdue_Layer")

col_exposure  = find_col(df, "Exposure")
col_impact    = find_col(df, "Business impact", "Impact")
col_lifetime  = find_col(df, "Data lifetime", "Lifetime")
col_pii       = find_col(df, "Contains PII", "PII")

col_fw        = find_col(df, "Firmware upgradable", "Firmware_upgradable")
col_cm        = find_col(df, "Crypto modifiable", "Crypto_modifiable")
col_vendor    = find_col(df, "Vendor Support of PQC", "Vendor support of PQC", "Vendor_Support_of_PQC")

col_place     = find_col(df, "Sec_Placement", "Sec Placement")
col_devtype   = find_col(df, "Device type", "Device_type", "DeviceType")

# ---- E, I, L (Risk components) ----
if col_exposure:
    E = df[col_exposure].apply(lambda x: exposure_score_from_protocols(x))
else:
    E = df[col_protocols].apply(exposure_score_from_protocols) if col_protocols else pd.Series(0.4, index=df.index)

if col_impact:
    I = df[col_impact].apply(business_impact_score)
    # fallback missing values -> purdue-based proxy
    if col_purdue:
        I = I.fillna(df[col_purdue].apply(purdue_layer_to_impact))
    else:
        I = I.fillna(0.6)
else:
    I = df[col_purdue].apply(purdue_layer_to_impact) if col_purdue else pd.Series(0.6, index=df.index)

if col_lifetime:
    L = df[col_lifetime].apply(lifetime_score).fillna(0.5)
else:
    L = pd.Series(0.5, index=df.index)  # default if you don't have lifetime column

# Optional PII boost (OFF by default)
include_pii = False
if include_pii and col_pii:
    P = df[col_pii].apply(lambda x: 1.0 if as_text(x).lower() in {"y","yes","true","1"} else 0.0)
    R = (0.45*E + 0.35*I + 0.20*L + 0.10*P) / 1.10
else:
    R = 0.45*E + 0.35*I + 0.20*L

# ---- Feasibility F ----
FW = df[col_fw].apply(yn_maybe) if col_fw else pd.Series(0.5, index=df.index)
CM = df[col_cm].apply(yn_maybe) if col_cm else pd.Series(0.5, index=df.index)

# Vendor: Y=1, Unknown=0.5, N=0
def vendor_score(x):
    s = as_text(x).lower()
    if s in {"y","yes","true","1"}: return 1.0
    if "unknown" in s or s == "" or "n/unknown" in s: return 0.5
    if s in {"n","no","false","0"}: return 0.0
    return 0.5

V = df[col_vendor].apply(vendor_score) if col_vendor else pd.Series(0.5, index=df.index)

F = 0.4*FW + 0.4*CM + 0.2*V

# ---- Cost/Friction C ----
S  = df[col_place].apply(placement_friction) if col_place else pd.Series(0.6, index=df.index)
Pu = df[col_purdue].apply(purdue_friction) if col_purdue else pd.Series(0.7, index=df.index)
D  = df[col_devtype].apply(device_friction) if col_devtype else pd.Series(0.7, index=df.index)

C = 0.4*S + 0.3*Pu + 0.3*D

# ---- PriorityScore = 0.5R + 0.3F - 0.2C ----
alpha, beta, gamma = 0.5, 0.3, 0.2
score = alpha*R + beta*F - gamma*C

level = np.select(
    [score >= 0.5, score >= 0.3],
    ["High", "Medium"],
    default="Low"
)

# ---- PRINT (no new df columns) ----
show_cols = []

col_id = find_col(df, "ID")
col_name = find_col(df, "Conduit Name", "Conduit_Name")
if col_id: show_cols.append(col_id)
if col_name: show_cols.append(col_name)
if col_purdue: show_cols.append(col_purdue)
if col_place: show_cols.append(col_place)

out = pd.DataFrame({
    **{c: df[c] for c in show_cols},
    "R": R, "F": F, "C": C,
    "PriorityScore": score,
    "PriorityLevel": level
})

print(out.head(25).to_string(index=False))

 ID                                             Conduit Name Purdue Layer                                       Sec_Placement     R   F    C  PriorityScore PriorityLevel
 C1                    Spot Market ↔ Energy Market Subsystem        L5–L4              EndToEnd (optionally also Gateway/VPN) 1.000 0.9 0.73         0.6240          High
 C2                            Energy Market Subsystem ↔ DMS        L4–L3                    EndToEnd (often also inside VPN) 0.865 0.9 0.82         0.5385          High
 C3                           Energy Market Subsystem ↔ MDMS        L4–L3                          EndToEnd (plus VPN tunnel) 0.865 0.9 0.82         0.5385          High
 C4                        SCADA System ↔ Automation Headend        L3–L3 GatewayOnly (between routers, not inside 104/61850) 0.765 0.9 0.66         0.5205          High
 C5                                 DMS ↔ Automation Headend        L3–L3                                         GatewayOnly 0.865 0.9 0.66         0

In [597]:
df["Migration_Priority"] = [
    f"{s:.3f} ({lvl})" if pd.notna(s) else np.nan
    for s, lvl in zip(score, level)
]

In [598]:
df

,ID,Conduit Name,Zones,Purdue Layer,Protocols,Sec_Protocol,Has_Enc,Auth_Type,Has_Int,Sec_Placement,...,Vendor Support of PQC,Possible Deployed CAT,Enc_Ranking_By_Utility,Enc_Best_Alg,Auth_Ranking_By_Utility,Auth_Best_Alg,Auth_Best_Utility,Enc_Confidence_Top3,Auth_Confidence_Top3,Migration_Priority
0,C1,Spot Market ↔ Energy Market Subsystem,Market – Energy Market Subsystem,L5–L4,"WebService, Internet, phone call",TLS 1.2/1.3 (sometimes plus VPN/IPsec),Yes,ServerCert,Yes,EndToEnd (optionally also Gateway/VPN),...,N/Unknown,5,"Kyber1024:0.06282556, ML-KEM-1024:0.06288374, ...",Kyber1024,"Falcon-padded-1024:0.06271120, Falcon-1024:0.0...",Falcon-padded-1024,0.062711,"Kyber1024:0.523135, ML-KEM-1024:0.476865, BIKE...","Falcon-padded-1024:0.996982, Falcon-1024:0.003...",0.624 (High)
1,C2,Energy Market Subsystem ↔ DMS,Energy Market – Grid Operation,L4–L3,WebService or Email,"TLS 1.2/1.3 (for WebService), TLS for SMTP + o...",Yes,ServerCert (sometimes Password for mail),Yes,EndToEnd (often also inside VPN),...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.06646480, Falcon-1024:0.0...",Falcon-padded-1024,0.066465,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
2,C3,Energy Market Subsystem ↔ MDMS,Energy Market – Metering,L4–L3,WebService or Email,TLS 1.2/1.3 (often via VPN),Yes,ServerCert,Yes,EndToEnd (plus VPN tunnel),...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.06646480, Falcon-1024:0.0...",Falcon-padded-1024,0.066465,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
3,C4,SCADA System ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850 over Ethernet","IPsec ESP (site-to-site), sometimes None in le...","Yes (where IPsec is used), No (legacy)",MutCert (site-to-site VPN),Yes (ESP auth / AEAD),"GatewayOnly (between routers, not inside 104/6...",...,N/Unknown,3,"Kyber768:0.04575275, ML-KEM-768:0.04588339, BI...",Kyber768,"Dilithium3:0.21349891, ML-DSA-65:0.22259312, c...",Dilithium3,0.213499,"Kyber768:0.570903, ML-KEM-768:0.429097, BIKE-L...","Dilithium3:0.986069, ML-DSA-65:0.013931, cross...",0.521 (High)
4,C5,DMS ↔ Automation Headend,Grid Operation,L3–L3,"IEC 60870-5-104, IEC 61850, WebService/Ethernet",IPsec ESP (site-to-site),Yes,MutCert,Yes,GatewayOnly,...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.13292961, Falcon-1024:0.1...",Falcon-padded-1024,0.132930,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.571 (High)
5,C6,AMI Headend ↔ MDMS,Metering,L3–L3,WebService over Ethernet,TLS 1.2/1.3,Yes,MutCert (server + client certs),Yes (AEAD or MAC),EndToEnd,...,N/Unknown,5,"Kyber1024:0.06314225, ML-KEM-1024:0.06337498, ...",Kyber1024,"Falcon-padded-1024:0.13292961, Falcon-1024:0.1...",Falcon-padded-1024,0.132930,"Kyber1024:0.591116, ML-KEM-1024:0.408884, BIKE...","Falcon-padded-1024:0.995767, Falcon-1024:0.004...",0.538 (High)
6,C7,Automation Headend ↔ Automation Frontend,Grid Operation – Station,L3–L2,"IEC 60870-5-104, IEC 61850, WebService over Et...","IPsec ESP (typical), sometimes None / MAC-only",Yes (when IPsec enabled),MutCert,Yes,GatewayOnly (WAN/radio edge),...,N,3,"Kyber768:0.04700373, ML-KEM-768:0.04765697, BI...",Kyber768,"Dilithium3:0.22829456, ML-DSA-65:0.27120560, c...",Dilithium3,0.228295,"Kyber768:0.800554, ML-KEM-768:0.199446, BIKE-L...","Dilithium3:1.000000, ML-DSA-65:0.000000, cross...",0.352 (Medium)
7,C8,Automation Frontend ↔ Primary Substation Node,Primary Substation (high/medium voltage),L2–L2,"IEC 60870-5-104, IEC 61850, WebService Ethernet",None,"Partial (Yes where MACsec/VPN used, No otherwise)",None (or link-layer keys in MACsec),Partial,LAN-only (inside substation),...,N,1,"ml-kem-512:0.48480465, bikel1:72.16349553, hqc...",ml-kem-512,,,0.000000,"ml-k